In [ ]:
!git clone https://github.com/tsinghua-fib-lab/AgentSocietyChallenge.git
%cd AgentSocietyChallenge

# 查看 Python 版本及项目文件
!python --version
!ls

In [ ]:
from pathlib import Path

baseline = Path("example/RecAgent_baseline.py")

print("Baseline exists:", baseline.exists())

for keyword in ["RecommendationAgent", "get_user", "get_item", "get_reviews", "evaluate", "InfinigenceLLM"]:
    print(f"{keyword}:", keyword in baseline.read_text())

print("\nRecommendation-related files:")
for p in Path("websocietysimulator").rglob("*.py"):
    if "recommend" in p.name.lower() or "evaluat" in p.name.lower():
        print(p)

In [ ]:
from pathlib import Path

# 确保位于仓库目录
repo = Path("/content/AgentSocietyChallenge")

files = { "Baseline Agent": repo / "example/RecAgent_baseline.py", "Recommendation Task": repo / "websocietysimulator/tasks/recommendation_task.py", "Recommendation Agent": repo / "websocietysimulator/agent/recommendation_agent.py",}

for name, path in files.items():
    print(f"\n{'='*20} {name} {'='*20}")

    if not path.exists():
        print("File not found:", path)
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines, start=1):
        if any(keyword in line for keyword in [
            "class ",
            "def ",
            "get_user(",
            "get_item(",
            "get_reviews(",
            "evaluate(",
            "simulator",
            "api_key",
            "model="
        ]):
            print(f"{i}: {line.strip()}")

In [ ]:
from pathlib import Path
import json
import inspect

repo = Path("/content/AgentSocietyChallenge")

# 1. 查看 RecommendationTask 的完整定义
task_file = repo / "websocietysimulator/tasks/recommendation_task.py"

print("=== RECOMMENDATION TASK FORMAT ===")
print(task_file.read_text(encoding="utf-8")[:5000])

# 2. 查看仓库中实际存在的任务文件
print("\n=== SAMPLE TASK FILES ===")

task_files = [p for p in repo.rglob("*.json")
    if "task" in str(p).lower()
    and "groundtruth" not in str(p).lower()
]

for p in task_files[:10]:
    print(p.relative_to(repo))

# 3. 查看 Baseline 的数据加载和运行部分
baseline = repo / "example/RecAgent_baseline.py"
lines = baseline.read_text(encoding="utf-8").splitlines()

print("\n=== BASELINE WORKFLOW ===")

for i in range(85, min(185, len(lines))):
    print(f"{i+1}: {lines[i]}")

In [ ]:
{"user_id": "...", "candidate_category": "...", "candidate_list": ["item1", "item2", "..."], "loc": [-1, -1]}

In [ ]:
from pathlib import Path
import json

repo = Path("/content/AgentSocietyChallenge")

# 1. 查找 Amazon 任务和 groundtruth
print("=== AMAZON TASK DIRECTORIES ===")

for p in repo.rglob("amazon"):
    if p.is_dir():
        print(p.relative_to(repo))

# 2. 检查官方示例任务结构
task_files = sorted((repo / "example/track2").glob("*/tasks/*.json"))

if task_files:
    sample = task_files[0]

    print("\n=== SAMPLE TASK ===")
    print("File:", sample.relative_to(repo))

    data = json.loads(sample.read_text())

    print(json.dumps(data, indent=2, ensure_ascii=False)[:2500])

# 3. 查看 Amazon 相关文件
print("\n=== AMAZON FILES ===")

amazon_files = [ p for p in repo.rglob("*") if p.is_file() and "amazon" in str(p).lower()]

for p in amazon_files[:25]:
    print(p.relative_to(repo))

print("\nTotal Amazon-related files:",
      len(amazon_files))

In [ ]:
from pathlib import Path
import json

repo = Path("/content/AgentSocietyChallenge")
amazon = repo / "example/track2/amazon"

tasks = sorted((amazon / "tasks").glob("*.json"))

groundtruth = sorted((amazon / "groundtruth").glob("*.json"))

print("Amazon tasks:", len(tasks))
print("Groundtruth files:", len(groundtruth))

print("\n=== SAMPLE GROUNDTRUTH ===")

if groundtruth:
    sample = groundtruth[0]
    print("File:", sample.name)
    print(sample.read_text()[:1500])

print("\n=== TASK USER IDS ===")

task_users = set()

for path in tasks:
    task = json.loads(path.read_text())
    task_users.add(task["user_id"])

print("Unique task users:", len(task_users))

if "shift_df" in globals():
    tvd_users = set(shift_df["user_id"].astype(str))

    matched = task_users & tvd_users

    print("TVD users:", len(tvd_users))
    print("Matched users:", len(matched))
else:
    print("shift_df not loaded in this Notebook.")

In [ ]:
from google.colab import files
import pandas as pd
import json
from pathlib import Path

uploaded = files.upload()

tvd_df = pd.read_csv("tvd_internal_user_mapping.csv",dtype={"user_id": str})

repo = Path("/content/AgentSocietyChallenge")
task_dir = repo / "example/track2/amazon/tasks"
task_records = []

for path in task_dir.glob("task_*.json"):
    task = json.loads(path.read_text())

    task_records.append({
        "task_id": path.stem,
        "user_id": str(task["user_id"]),
        "n_candidates": len(task["candidate_list"])
    })

tasks_df = pd.DataFrame(task_records)

matched_df = tasks_df.merge(tvd_df, on="user_id", how="inner")

print("Total recommendation tasks:", len(tasks_df))
print("TVD users:", tvd_df["user_id"].nunique())
print("Matched users:", matched_df["user_id"].nunique())
print("Matched tasks:", len(matched_df))

display(matched_df.head())

In [ ]:
from pathlib import Path

repo = Path("/content/AgentSocietyChallenge")

print("=== DATA FILES ===")

extensions = {".csv", ".jsonl", ".parquet", ".db", ".sqlite"}

data_files = [
    p for p in repo.rglob("*")
    if p.is_file()
    and p.suffix.lower() in extensions
]

for p in data_files[:60]:
    print(p.relative_to(repo))

print("\nTotal data files:", len(data_files))

print("\n=== DATA DIRECTORY STRUCTURE ===")

for name in ["example", "websocietysimulator", "data_images"]:
    folder = repo / name
    if folder.exists():
        print(f"\n{name}/")
        for p in list(folder.iterdir())[:20]:
            print("  ", p.name)

In [ ]:
from pathlib import Path

repo = Path("/content/AgentSocietyChallenge")

files_to_check = [
    repo / "websocietysimulator/simulator.py",
    repo / "websocietysimulator/tools/interaction_tool.py",
    repo / "README.md"
]

keywords = [
    "data_dir",
    "download",
    "dataset",
    "get_reviews",
    "review",
    "timestamp",
    "date",
    "huggingface",
    "track2"
]

for path in files_to_check:
    print(f"\n{'='*20} {path.name} {'='*20}")

    if not path.exists():
        print("File not found")
        continue

    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    for i, line in enumerate(lines, 1):
        if any(
            keyword.lower() in line.lower()
            for keyword in keywords
        ):
            print(f"{i}: {line.strip()[:180]}")

In [ ]:
import requests

repo_id = "SGJQovo/AgentRecBench"

url = (
    f"https://huggingface.co/api/datasets/"
    f"{repo_id}/tree/main"
)

all_files = []
cursor = url

while cursor:
    response = requests.get(
        cursor,
        params={
            "recursive": "true",
            "limit": 1000
        } if cursor == url else None,
        timeout=60
    )
    response.raise_for_status()

    all_files.extend(response.json())

    # Follow Hugging Face pagination if present
    next_url = None
    for part in response.headers.get(
        "Link", ""
    ).split(","):
        if 'rel="next"' in part:
            next_url = part.split(";")[0].strip(" <>")
            break

    cursor = next_url

print("=== AMAZON FILES ===")

amazon_files = [
    f for f in all_files
    if f["type"] == "file"
    and "amazon" in f["path"].lower()
]

for f in amazon_files:
    size_mb = f.get("size", 0) / (1024**2)
    print(
        f"{f['path']} | "
        f"{size_mb:.2f} MB"
    )

print("\nTotal Amazon files:", len(amazon_files))

In [ ]:
import pandas as pd
import requests
from io import StringIO

base = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "task/long_term/amazon/"
)

files_to_check = [
    "amazon.csv",
    "amazon_case.csv",
    "amazon_sample.csv"
]

for filename in files_to_check:
    response = requests.get(
        base + filename,
        timeout=60
    )
    response.raise_for_status()

    df = pd.read_csv(
        StringIO(response.text)
    )

    print(f"\n=== {filename} ===")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if "user_id" in df.columns:
        counts = df.groupby("user_id").size()

        print("Unique users:", len(counts))
        print("Median interactions:", counts.median())
        print("Mean interactions:", round(counts.mean(), 2))
        print("Users with >= 4 interactions:",
              (counts >= 4).sum())
        print("Users with >= 10 interactions:",
              (counts >= 10).sum())

    display(df.head(2))

In [ ]:
full_df = pd.read_csv(base + "amazon.csv")
case_df = pd.read_csv(base + "amazon_case.csv")
sample_df = pd.read_csv(base + "amazon_sample.csv")

keys = ["user_id", "item_id", "date"]

case_users = set(case_df["user_id"])
sample_users = set(sample_df["user_id"])

print("=== USER OVERLAP ===")
print("Case users:", len(case_users))
print("Sample users:", len(sample_users))
print("Overlapping users:", len(case_users & sample_users))

# Identify records in sample but not case,
# accounting for possible duplicate rows
case_counts = (
    case_df.groupby(keys, dropna=False)
    .size()
    .rename("case_n")
)
sample_counts = (
    sample_df.groupby(keys, dropna=False)
    .size()
    .rename("sample_n")
)

comparison = pd.concat(
    [case_counts, sample_counts],
    axis=1
).fillna(0)

comparison["extra_n"] = (
    comparison["sample_n"] - comparison["case_n"]
).clip(lower=0)

extra = comparison[
    comparison["extra_n"] > 0
].reset_index()

print("\n=== SAMPLE-ONLY INTERACTIONS ===")
print("Total extra records:", int(extra["extra_n"].sum()))
print("Users with extra records:", extra["user_id"].nunique())

print("\nExtra records per user:")
print(extra.groupby("user_id")["extra_n"].sum().value_counts())

print("\n=== FULL DATA COVERAGE ===")
print(
    "Case users found in full data:",
    len(case_users & set(full_df["user_id"]))
)

display(extra.head())

In [ ]:
import pandas as pd

# Reconstruct the 100 held-out interactions
extra_targets = extra.loc[
    extra.index.repeat(extra["extra_n"].astype(int)),
    ["user_id", "item_id", "date"]
].copy()

# Latest timestamp in each user's case history
case_last = (
    case_df.groupby("user_id")["date"]
    .max()
    .rename("last_history_date")
)

target_check = extra_targets.merge(
    case_last,
    on="user_id",
    how="left"
)

target_check["is_future"] = (
    target_check["date"] >
    target_check["last_history_date"]
)

print("=== TEMPORAL VALIDATION ===")
print("Total targets:", len(target_check))
print("Targets after all history:",
      target_check["is_future"].sum())
print("Targets not after all history:",
      (~target_check["is_future"]).sum())

print("\n=== EXAMPLES ===")
display(target_check.head())

print("\n=== FULL DATA CHECK ===")

# Check whether each target also appears
# in the full interaction dataset
target_in_full = extra_targets.merge(
    full_df[["user_id", "item_id", "date"]],
    on=["user_id", "item_id", "date"],
    how="left",
    indicator=True
)

print(target_in_full["_merge"].value_counts())

In [ ]:
import requests
from collections import Counter

repo = "SGJQovo/AgentRecBench"
root = (
    f"https://huggingface.co/datasets/{repo}/"
    "resolve/main/task/long_term/amazon/"
)

official_records = []

for i in range(100):
    task_url = root + f"tasks/task_{i}.json"
    gt_url = root + f"groundtruth/groundtruth_{i}.json"

    task_resp = requests.get(task_url, timeout=30)
    gt_resp = requests.get(gt_url, timeout=30)

    task_resp.raise_for_status()
    gt_resp.raise_for_status()

    task = task_resp.json()
    gt = gt_resp.json()

    official_records.append({
        "task_id": i,
        "task": task,
        "groundtruth": gt
    })

print("=== EXAMPLE TASK ===")
print(official_records[0])

print("\n=== TASK FIELDS ===")
print(Counter(
    tuple(sorted(r["task"].keys()))
    for r in official_records
))

print("\n=== GROUNDTRUTH FIELDS ===")
print(Counter(
    tuple(sorted(r["groundtruth"].keys()))
    for r in official_records
))

In [ ]:
target_map = dict(zip(
    extra_targets["user_id"],
    extra_targets["item_id"]
))

results = []

for record in official_records:
    task = record["task"]
    gt = record["groundtruth"]["ground truth"]

    user = task["user_id"]
    candidates = task["candidate_list"]
    heldout = target_map.get(user)

    results.append({
        "task_id": record["task_id"],
        "user_id": user,
        "candidate_count": len(candidates),
        "groundtruth_matches_heldout": gt == heldout,
        "groundtruth_in_candidates": gt in candidates,
        "unique_candidates": len(set(candidates)) == len(candidates),
        "has_case_history": user in case_users
    })

validation_df = pd.DataFrame(results)

print("=== OFFICIAL TASK VALIDATION ===")
print("Tasks:", len(validation_df))
print("Unique users:", validation_df["user_id"].nunique())

for col in [
    "groundtruth_matches_heldout",
    "groundtruth_in_candidates",
    "unique_candidates",
    "has_case_history"
]:
    print(f"{col}: {validation_df[col].sum()}/100")

print("\nCandidate count distribution:")
print(validation_df["candidate_count"].value_counts())

print("\nProblematic tasks:")
display(validation_df[
    ~validation_df[
        [
            "groundtruth_matches_heldout",
            "groundtruth_in_candidates",
            "unique_candidates",
            "has_case_history"
        ]
    ].all(axis=1)
])

In [ ]:
import pandas as pd

df = full_df.copy()

# Sort chronologically within each user
df = df.sort_values(
    ["user_id", "date", "item_id"]
).reset_index(drop=True)

# Assign interaction order
df["interaction_index"] = (
    df.groupby("user_id").cumcount()
)

df["total_interactions"] = (
    df.groupby("user_id")["item_id"].transform("size")
)

# Last interaction becomes target
is_target = (
    df["interaction_index"] ==
    df["total_interactions"] - 1
)

expanded_targets = df[is_target].copy()
expanded_history = df[~is_target].copy()

# Count history interactions
history_counts = (
    expanded_history.groupby("user_id")
    .size()
    .rename("history_length")
)

expanded_targets = expanded_targets.merge(
    history_counts,
    on="user_id",
    how="left"
)

expanded_targets["history_length"] = (
    expanded_targets["history_length"].fillna(0).astype(int)
)

print("=== EXPANDED DATASET ===")
print("Users:", expanded_targets["user_id"].nunique())
print("History interactions:", len(expanded_history))
print("Held-out targets:", len(expanded_targets))

print("\n=== HISTORY LENGTH THRESHOLDS ===")
for threshold in [4, 6, 8, 10]:
    n = (expanded_targets["history_length"] >= threshold).sum()
    print(f"History >= {threshold}: {n} users")

print("\n=== HISTORY LENGTH DISTRIBUTION ===")
display(
    expanded_targets["history_length"]
    .describe()
)

print("\n=== EXAMPLE TARGETS ===")
display(
    expanded_targets[
        ["user_id", "item_id", "date", "history_length"]
    ].head()
)

In [ ]:
longterm_items = set(full_df["item_id"])
history_items = set(expanded_history["item_id"])
target_items = set(expanded_targets["item_id"])

print("=== LONG-TERM AMAZON ITEMS ===")
print("Unique items (all):", len(longterm_items))
print("Unique history items:", len(history_items))
print("Unique target items:", len(target_items))

print("\n=== HISTORY-TARGET OVERLAP ===")
print(
    "Target items also appearing in history pool:",
    len(target_items & history_items)
)

print("\n=== OFFICIAL CANDIDATE COVERAGE ===")
official_candidates = set()

for record in official_records:
    official_candidates.update(
        record["task"]["candidate_list"]
    )

print("Unique official candidate items:",
      len(official_candidates))
print("Candidates found in long-term interactions:",
      len(official_candidates & longterm_items))

print("\n=== ITEM ID EXAMPLES ===")
print(sorted(list(longterm_items))[:10])

In [ ]:
import requests

api_url = (
    "https://huggingface.co/api/datasets/"
    "SGJQovo/AgentRecBench/tree/main"
)

def list_repo_files(path="", recursive=True):
    url = api_url + ("/" + path if path else "")
    response = requests.get(
        url,
        params={
            "recursive": str(recursive).lower(),
            "expand": "false"
        },
        timeout=60
    )
    response.raise_for_status()
    return response.json()

# Use the dataset info endpoint to obtain the full file list
info = requests.get(
    "https://huggingface.co/api/datasets/"
    "SGJQovo/AgentRecBench",
    timeout=60
).json()

all_paths = [
    entry["rfilename"]
    for entry in info.get("siblings", [])
]

keywords = [
    "meta",
    "item.json",
    "product",
    "category",
    "asin",
    "item_info"
]

matches = [
    p for p in all_paths
    if any(k in p.lower() for k in keywords)
]

print("=== METADATA CANDIDATES ===")
print("Total repository files:", len(all_paths))
print("Matching paths:", len(matches))

for path in matches[:100]:
    print(path)

In [ ]:
import requests

repo = "SGJQovo/AgentRecBench"

paths = [
    "output_data_long/item.json",
    "output_data_all/item.json",
    "output_data_short/item.json"
]

# File sizes from Hugging Face repository tree
tree_url = (
    f"https://huggingface.co/api/datasets/{repo}/tree/main"
    "?recursive=true"
)

response = requests.get(tree_url, timeout=60)
response.raise_for_status()
tree = response.json()

size_map = {
    f["path"]: f.get("size")
    for f in tree
    if f.get("type") == "file"
}

for path in paths:
    print(f"\n=== {path} ===")

    size = size_map.get(path)
    print(
        "Size (MB):",
        round(size / 1024**2, 2)
        if size is not None else "Unknown"
    )

    url = (
        f"https://huggingface.co/datasets/{repo}"
        f"/resolve/main/{path}"
    )

    # Request only the first 2 KB
    r = requests.get(
        url,
        headers={"Range": "bytes=0-2047"},
        timeout=60,
        stream=True
    )
    r.raise_for_status()

    preview = b""
    for chunk in r.iter_content(chunk_size=2048):
        preview += chunk
        if len(preview) >= 2048:
            break
    r.close()

    print("Preview:")
    print(preview[:800].decode("utf-8", errors="replace"))

In [ ]:
import requests

url = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "output_data_long/item.json"
)

# Check whether server supports byte-range requests
head = requests.head(
    url,
    allow_redirects=True,
    timeout=60
)

print("=== FILE HEADERS ===")
print("Status:", head.status_code)
print("Content-Length:", head.headers.get("Content-Length"))
print("Accept-Ranges:", head.headers.get("Accept-Ranges"))

# Request the final 16 KB
r = requests.get(
    url,
    headers={"Range": "bytes=-16384"},
    stream=True,
    timeout=60
)

print("\n=== RANGE RESPONSE ===")
print("Status:", r.status_code)
print("Content-Range:", r.headers.get("Content-Range"))

if r.status_code == 206:
    content = r.content.decode("utf-8", errors="replace")
    lines = content.splitlines()

    print("\n=== LAST 3 RECORDS ===")
    for line in lines[-3:]:
        print(line[:1000])
        print("---")
else:
    print(
        "Server did not honor the range request. "
        "Skipping download."
    )
    r.close()

In [ ]:
import requests
import json

url = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "output_data_long/item.json"
)

file_size = 1360464643
chunk_size = 16384

for fraction in [0.25, 0.50, 0.75]:
    start = int(file_size * fraction)
    end = start + chunk_size - 1

    response = requests.get(
        url,
        headers={"Range": f"bytes={start}-{end}"},
        timeout=60
    )

    print(f"\n=== POSITION {fraction:.0%} ===")
    print("HTTP status:", response.status_code)

    if response.status_code != 206:
        print("Range request not supported here.")
        continue

    # The first and last lines may be incomplete.
    # Parse only complete JSONL records.
    lines = response.content.split(b"\n")[1:-1]

    parsed = []
    for line in lines:
        try:
            parsed.append(json.loads(line))
        except (json.JSONDecodeError, UnicodeDecodeError):
            continue

    print("Complete records parsed:", len(parsed))

    for item in parsed[:3]:
        print({
            "item_id": item.get("item_id"),
            "source": item.get("source"),
            "title": item.get("title", item.get("name")),
            "categories": item.get("categories"),
            "category": item.get("category")
        })

In [ ]:
# Step 19: Find existing metadata DataFrames in Colab

import pandas as pd

candidates = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = [str(c).lower() for c in obj.columns]

        if any(
            keyword in " ".join(cols)
            for keyword in [
                "category",
                "categories",
                "title",
                "parent_asin",
                "asin"
            ]
        ):
            candidates.append({
                "variable": name,
                "rows": len(obj),
                "columns": obj.columns.tolist()[:15]
            })

print("=== POSSIBLE METADATA TABLES ===")

if candidates:
    display(pd.DataFrame(candidates))
else:
    print("No existing metadata DataFrame found.")

In [ ]:
import requests
import json
import pandas as pd
from collections import Counter

url = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "output_data_long/item.json"
)

file_size = 1360464643
chunk_size = 8192

session = requests.Session()
results = []

for pct in range(0, 100, 2):
    start = int(file_size * pct / 100)
    end = min(start + chunk_size - 1, file_size - 1)

    try:
        r = session.get(
            url,
            headers={"Range": f"bytes={start}-{end}"},
            timeout=30
        )
        r.raise_for_status()

        if r.status_code != 206:
            print("Range not honored; stopping.")
            break

        # Discard potentially incomplete boundary lines
        lines = r.content.split(b"\n")[1:-1]
        sources = []

        for line in lines:
            try:
                obj = json.loads(line)
                sources.append(
                    obj.get("source", "unknown")
                )
            except json.JSONDecodeError:
                continue

        most_common = (
            Counter(sources).most_common(1)
        )

        source = (
            most_common[0][0]
            if most_common else "unparsed"
        )

        results.append({
            "position_pct": pct,
            "source": source,
            "parsed_records": len(sources)
        })

    except requests.RequestException as e:
        print(f"Error at {pct}%: {e}")

scan_df = pd.DataFrame(results)

print("=== SOURCE DISTRIBUTION ===")
print(scan_df["source"].value_counts())

print("\n=== NON-GOODREADS REGIONS ===")
display(
    scan_df[
        scan_df["source"] != "goodreads"
    ]
)

print("\n=== AMAZON REGIONS ===")
display(
    scan_df[
        scan_df["source"].astype(str).str.contains(
            "amazon", case=False
        )
    ]
)

In [ ]:
import requests
import json

url = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "output_data_long/item.json"
)

file_size = 1360464643

start = int(file_size * 0.10)
end = start + 65535

r = requests.get(
    url,
    headers={"Range": f"bytes={start}-{end}"},
    timeout=60
)

assert r.status_code == 206, "Range request failed"

# Remove potentially incomplete first and last lines
lines = r.content.split(b"\n")[1:-1]

amazon_items = []

for line in lines:
    try:
        item = json.loads(line)
        if item.get("source") == "amazon":
            amazon_items.append(item)
    except json.JSONDecodeError:
        continue

print("=== AMAZON RECORDS FOUND ===")
print(len(amazon_items))

for item in amazon_items[:3]:
    print("\n=== ITEM ===")
    print("Item ID:", item.get("item_id"))
    print("Title:", item.get("title"))
    print("Categories:", item.get("categories"))
    print("Main category:", item.get("main_category"))
    print("Category:", item.get("category"))
    print("Description:", str(item.get("description"))[:250])
    print("Available fields:", list(item.keys()))

In [ ]:
# Step 22: Extract required Amazon metadata
# using HTTP Range requests

import requests
import json
import pandas as pd

url = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "output_data_long/item.json"
)

needed_ids = longterm_items | official_candidates

file_size = 1360464643

# Conservative scan: first 25% of the file
scan_end = int(file_size * 0.25)

chunk_size = 4 * 1024 * 1024
session = requests.Session()

metadata = {}
buffer = b""
position = 0
total_amazon = 0
total_parsed = 0

while position < scan_end:
    end = min(
        position + chunk_size - 1,
        scan_end - 1
    )

    response = session.get(
        url,
        headers={
            "Range": f"bytes={position}-{end}"
        },
        timeout=120
    )
    response.raise_for_status()

    if response.status_code != 206:
        raise RuntimeError(
            "Server did not honor byte-range request"
        )

    data = buffer + response.content
    lines = data.split(b"\n")

    # Preserve incomplete final JSONL record
    buffer = lines.pop()

    for line in lines:
        if not line.strip():
            continue

        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            continue

        total_parsed += 1

        if item.get("source") != "amazon":
            continue

        total_amazon += 1
        item_id = str(item.get("item_id", ""))

        if item_id in needed_ids:
            metadata[item_id] = {
                "item_id": item_id,
                "main_category": item.get(
                    "main_category"
                ),
                "categories": item.get(
                    "categories"
                ),
                "title": item.get("title"),
                "description": item.get(
                    "description"
                )
            }

    position = end + 1

    if position % (40 * 1024 * 1024) < chunk_size:
        print(
            f"Scanned {position / 1024**2:.0f} MB | "
            f"Matched {len(metadata)} items"
        )

# Parse the final buffered record if complete
if buffer.strip():
    try:
        item = json.loads(buffer)
        if item.get("source") == "amazon":
            total_amazon += 1
            item_id = str(item.get("item_id", ""))
            if item_id in needed_ids:
                metadata[item_id] = {
                    "item_id": item_id,
                    "main_category": item.get(
                        "main_category"
                    ),
                    "categories": item.get(
                        "categories"
                    ),
                    "title": item.get("title"),
                    "description": item.get(
                        "description"
                    )
                }
    except json.JSONDecodeError:
        pass

metadata_df = pd.DataFrame(
    list(metadata.values())
)

print("\n=== EXTRACTION RESULTS ===")
print("Parsed records:", total_parsed)
print("Amazon records:", total_amazon)
print("Needed unique items:", len(needed_ids))
print("Matched items:", len(metadata_df))

print("\n=== COVERAGE ===")
print(
    "Long-term item coverage:",
    f"{len(longterm_items & set(metadata)) / len(longterm_items):.2%}"
)

print(
    "Official candidate coverage:",
    f"{len(official_candidates & set(metadata)) / len(official_candidates):.2%}"
)

display(metadata_df.head())

# Save locally in Colab
metadata_df.to_csv(
    "/content/amazon_metadata_matched.csv",
    index=False
)

print("\nSaved: /content/amazon_metadata_matched.csv")


In [ ]:
# Step 23: Inspect category quality and coverage

import pandas as pd
import ast
from collections import Counter

meta = metadata_df.copy()

def parse_categories(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            pass
    return []

meta["category_path"] = meta["categories"].apply(
    parse_categories
)

meta["level_1"] = meta["category_path"].apply(
    lambda x: x[0] if len(x) >= 1 else None
)

meta["level_2"] = meta["category_path"].apply(
    lambda x: x[1] if len(x) >= 2 else None
)

print("=== METADATA QUALITY ===")
print("Total items:", len(meta))

for col in ["main_category", "level_1", "level_2"]:
    valid = meta[col].notna().sum()
    print(
        f"{col}: {valid}/{len(meta)} "
        f"({valid / len(meta):.2%})"
    )
    print("Unique categories:", meta[col].nunique())

print("\n=== TOP LEVEL-1 CATEGORIES ===")
display(meta["level_1"].value_counts().head(15))

print("\n=== TOP LEVEL-2 CATEGORIES ===")
display(meta["level_2"].value_counts().head(20))

# Match category metadata to user history
history_with_cat = expanded_history.merge(
    meta[["item_id", "level_1", "level_2"]],
    on="item_id",
    how="left",
    validate="many_to_one"
)

print("\n=== HISTORY COVERAGE ===")
print("History interactions:", len(history_with_cat))

for col in ["level_1", "level_2"]:
    valid = history_with_cat[col].notna().sum()
    print(
        f"{col}: {valid}/{len(history_with_cat)} "
        f"({valid / len(history_with_cat):.2%})"
    )

# How many users have at least 3 valid category
# observations in both chronological halves?
history_with_cat = history_with_cat.sort_values(
    ["user_id", "date", "item_id"]
).copy()

history_with_cat["position"] = (
    history_with_cat.groupby("user_id").cumcount()
)

history_with_cat["n_history"] = (
    history_with_cat.groupby("user_id")["item_id"]
    .transform("size")
)

history_with_cat["period"] = (
    history_with_cat["position"] <
    history_with_cat["n_history"] // 2
).map({True: "early", False: "recent"})

for col in ["level_1", "level_2"]:
    counts = (
        history_with_cat
        .dropna(subset=[col])
        .groupby(["user_id", "period"])
        .size()
        .unstack(fill_value=0)
        .reindex(
            columns=["early", "recent"],
            fill_value=0
        )
    )

    eligible = (
        (counts["early"] >= 3) &
        (counts["recent"] >= 3)
    ).sum()

    print(
        f"\nUsers with >=3 valid {col} "
        f"observations in BOTH halves: {eligible}"
    )

In [ ]:
# Step 24: Compute user-level Raw and Excess TVD

import numpy as np
import pandas as pd
from collections import Counter

B = 500
rng = np.random.default_rng(2026)

def calculate_tvd(a, b):
    count_a = Counter(a)
    count_b = Counter(b)

    categories = set(count_a) | set(count_b)

    return 0.5 * sum(
        abs(
            count_a[c] / len(a)
            - count_b[c] / len(b)
        )
        for c in categories
    )

def measure_shift(values, B=500):
    values = np.asarray(values, dtype=object)

    n = len(values)
    split = n // 2

    early = values[:split]
    recent = values[split:]

    observed = calculate_tvd(early, recent)

    shuffled = np.empty(B)

    for b in range(B):
        perm = rng.permutation(values)
        shuffled[b] = calculate_tvd(
            perm[:split],
            perm[split:]
        )

    null_mean = shuffled.mean()
    excess = observed - null_mean

    p_value = (
        1 + np.sum(shuffled >= observed)
    ) / (B + 1)

    return observed, null_mean, excess, p_value

records = []

for user_id, group in history_with_cat.groupby(
    "user_id", sort=False
):
    group = group.sort_values(
        ["date", "item_id"]
    )

    row = {
        "user_id": user_id,
        "history_length": len(group)
    }

    for level in ["level_1", "level_2"]:
        valid_values = (
            group[level].dropna().tolist()
        )

        if len(valid_values) < 6:
            row[f"{level}_raw_tvd"] = np.nan
            row[f"{level}_null_mean"] = np.nan
            row[f"{level}_excess_tvd"] = np.nan
            row[f"{level}_pvalue"] = np.nan
            continue

        raw, null, excess, p = measure_shift(
            valid_values, B=B
        )

        row[f"{level}_raw_tvd"] = raw
        row[f"{level}_null_mean"] = null
        row[f"{level}_excess_tvd"] = excess
        row[f"{level}_pvalue"] = p

    records.append(row)

shift_df = pd.DataFrame(records)

print("=== MEASUREMENT SAMPLE ===")
print(
    "Users with Level-2 TVD:",
    shift_df["level_2_raw_tvd"].notna().sum()
)

print("\n=== LEVEL-2 RESULTS ===")
display(
    shift_df[
        [
            "level_2_raw_tvd",
            "level_2_null_mean",
            "level_2_excess_tvd",
            "level_2_pvalue"
        ]
    ].describe()
)

print("\n=== LEVEL-1 RESULTS ===")
display(
    shift_df[
        [
            "level_1_raw_tvd",
            "level_1_null_mean",
            "level_1_excess_tvd"
        ]
    ].describe()
)

print("\n=== CORRELATION ===")
print(
    shift_df[
        [
            "level_1_raw_tvd",
            "level_2_raw_tvd",
            "level_2_excess_tvd",
            "history_length"
        ]
    ].corr().round(3)
)

shift_df.to_csv(
    "/content/amazon_user_preference_shift.csv",
    index=False
)

print("\nSaved: /content/amazon_user_preference_shift.csv")

In [ ]:
# Step 25: Strict chronological TVD + bootstrap CI

import numpy as np
import pandas as pd
from collections import Counter

B = 500
rng = np.random.default_rng(2026)

def tvd(a, b):
    ca, cb = Counter(a), Counter(b)
    keys = set(ca) | set(cb)
    return 0.5 * sum(
        abs(ca[k] / len(a) - cb[k] / len(b))
        for k in keys
    )

results = []

for user_id, group in history_with_cat.groupby("user_id"):
    group = group.sort_values(
        ["date", "item_id"]
    )

    n = len(group)
    split = n // 2

    early = group.iloc[:split]["level_2"].dropna().tolist()
    recent = group.iloc[split:]["level_2"].dropna().tolist()

    if len(early) < 3 or len(recent) < 3:
        continue

    observed = tvd(early, recent)

    values = np.array(early + recent, dtype=object)
    n_early = len(early)

    null_values = np.empty(B)

    for b in range(B):
        shuffled = rng.permutation(values)
        null_values[b] = tvd(
            shuffled[:n_early],
            shuffled[n_early:]
        )

    null_mean = null_values.mean()

    results.append({
        "user_id": user_id,
        "history_length": n,
        "early_valid": len(early),
        "recent_valid": len(recent),
        "raw_tvd": observed,
        "null_mean": null_mean,
        "excess_tvd": observed - null_mean,
        "permutation_p": (
            1 + np.sum(null_values >= observed)
        ) / (B + 1)
    })

measurement_df = pd.DataFrame(results)

# User-level bootstrap for mean excess TVD
values = measurement_df["excess_tvd"].to_numpy()
rng_boot = np.random.default_rng(2027)

boot_means = np.array([
    rng_boot.choice(
        values, size=len(values), replace=True
    ).mean()
    for _ in range(5000)
])

ci_low, ci_high = np.quantile(
    boot_means, [0.025, 0.975]
)

print("=== STRICT MEASUREMENT RESULTS ===")
print("Users:", len(measurement_df))
print("Mean Raw TVD:", round(measurement_df["raw_tvd"].mean(), 4))
print("Mean Null TVD:", round(measurement_df["null_mean"].mean(), 4))
print("Mean Excess TVD:", round(values.mean(), 4))
print("95% Bootstrap CI:",
      (round(ci_low, 4), round(ci_high, 4)))

print("\n=== USER-LEVEL SIGNAL ===")
print(
    "Users with positive Excess TVD:",
    (measurement_df["excess_tvd"] > 1e-10).sum()
)
print(
    "Users with permutation p < 0.05:",
    (measurement_df["permutation_p"] < 0.05).sum()
)

measurement_df.to_csv(
    "/content/amazon_tvd_final.csv",
    index=False
)

print("\nSaved: /content/amazon_tvd_final.csv")

In [ ]:
history_with_cat.shape
history_with_cat.columns.tolist()
history_with_cat.head(10)

In [ ]:
measurement_df.shape
measurement_df.columns.tolist()
measurement_df.head()

In [ ]:
# Step 27.1: Filter eligible users

eligible_users = measurement_df["user_id"].unique()

behavior_df = history_with_cat[
    history_with_cat["user_id"].isin(eligible_users)
].copy()

print("=== ELIGIBLE BEHAVIORAL SAMPLE ===")
print("Eligible users:", len(eligible_users))
print("Unique users:", behavior_df["user_id"].nunique())
print("Total historical interactions:", len(behavior_df))

print("\n=== MISSING CATEGORY VALUES ===")
print(behavior_df[["level_1", "level_2"]].isna().sum())

assert behavior_df["user_id"].nunique() == len(eligible_users)

display(behavior_df.head())

In [ ]:
# Step 27.2: Construct valid category sequences

behavior_df = behavior_df.sort_values(
    ["user_id", "date", "item_id"]
).reset_index(drop=True)

valid_behavior_df = behavior_df.dropna(
    subset=["level_2"]
).copy()

category_sequences = (
    valid_behavior_df
    .groupby("user_id")["level_2"]
    .apply(list)
    .reset_index(name="category_sequence")
)

category_sequences["valid_history_length"] = (
    category_sequences["category_sequence"].apply(len)
)

print("=== CATEGORY SEQUENCES ===")
print("Total users:", len(category_sequences))
print("Valid interactions:", len(valid_behavior_df))
print("Removed interactions:",
      len(behavior_df) - len(valid_behavior_df))

display(category_sequences.head())

# Check consistency with Step 25

length_check = category_sequences.merge(
    measurement_df[
        ["user_id", "early_valid", "recent_valid"]
    ],
    on="user_id",
    validate="one_to_one"
)

assert len(length_check) == len(measurement_df)

assert (
    length_check["valid_history_length"]
    == length_check["early_valid"]
    + length_check["recent_valid"]
).all()

print("\nConsistency check: PASSED")

In [ ]:
# Step 27.3: Calculate consumer diversity metrics

import numpy as np
import pandas as pd

def calculate_diversity(sequence):

    counts = pd.Series(sequence).value_counts()

    probabilities = counts / counts.sum()

    category_count = len(counts)

    category_entropy = -np.sum(
        probabilities * np.log(probabilities)
    )

    normalized_entropy = (
        category_entropy / np.log(category_count)
        if category_count > 1 else 0.0
    )

    return pd.Series({
        "category_count": category_count,
        "category_entropy": category_entropy,
        "normalized_entropy": normalized_entropy
    })


diversity_metrics = (
    category_sequences["category_sequence"]
    .apply(calculate_diversity)
)

diversity_df = pd.concat(
    [
        category_sequences[
            ["user_id", "valid_history_length"]
        ],
        diversity_metrics
    ],
    axis=1
)

diversity_df["category_count"] = (
    diversity_df["category_count"].astype(int)
)

print("=== CONSUMER DIVERSITY DATASET ===")
print("Shape:", diversity_df.shape)

display(diversity_df.head(10))

In [ ]:
# Step 27.4: Validate and explore diversity measurements

import matplotlib.pyplot as plt

print("=== DATA QUALITY CHECK ===")

print("Total users:", len(diversity_df))
print("Unique users:", diversity_df["user_id"].nunique())
print("Duplicate users:",
      diversity_df["user_id"].duplicated().sum())

print("\nMissing values:")
print(diversity_df.isna().sum())

# Validate measurement boundaries

assert len(diversity_df) == len(measurement_df)
assert diversity_df["user_id"].is_unique
assert diversity_df.isna().sum().sum() == 0

assert (
    diversity_df["category_count"]
    <= diversity_df["valid_history_length"]
).all()

assert (
    diversity_df["category_entropy"] >= -1e-10
).all()

assert (
    diversity_df["category_entropy"]
    <= np.log(diversity_df["category_count"]) + 1e-10
).all()

assert diversity_df["normalized_entropy"].between(
    -1e-10, 1 + 1e-10
).all()

print("\nAll validation checks: PASSED")

# Descriptive statistics

print("\n=== DESCRIPTIVE STATISTICS ===")

display(
    diversity_df.drop(columns="user_id")
    .describe()
    .round(4)
)

# Merge with existing TVD measurements

diversity_analysis_df = measurement_df.merge(
    diversity_df,
    on="user_id",
    how="inner",
    validate="one_to_one"
)

# Preliminary correlation analysis

analysis_columns = [
    "history_length",
    "valid_history_length",
    "category_count",
    "category_entropy",
    "normalized_entropy",
    "raw_tvd",
    "excess_tvd"
]

print("\n=== SPEARMAN CORRELATION ===")

display(
    diversity_analysis_df[analysis_columns]
    .corr(method="spearman")
    .round(3)
)

# Distribution visualization

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = [
    ("category_count", "Category Count"),
    ("category_entropy", "Shannon Entropy"),
    ("normalized_entropy", "Normalized Entropy")
]

for ax, (column, title) in zip(axes, metrics):

    ax.hist(
        diversity_df[column],
        bins=25,
        color="steelblue",
        edgecolor="white",
        alpha=0.85
    )

    ax.axvline(
        diversity_df[column].median(),
        color="darkorange",
        linestyle="--",
        label="Median"
    )

    ax.set_title(title)
    ax.set_xlabel(title)
    ax.set_ylabel("Number of Users")
    ax.legend()

plt.suptitle("Distribution of Consumer Diversity Metrics")
plt.tight_layout()
plt.show()

In [ ]:
# Step 28.1: Identify valid adjacent interactions

persistence_base = behavior_df.sort_values(
    ["user_id", "date", "item_id"]
).reset_index(drop=True).copy()

# Retrieve previous interaction within each user
persistence_base["previous_category"] = (
    persistence_base.groupby("user_id")["level_2"]
    .shift(1)
)

persistence_base["previous_position"] = (
    persistence_base.groupby("user_id")["position"]
    .shift(1)
)

# Check whether two interactions are genuinely adjacent
persistence_base["is_adjacent"] = (
    persistence_base["position"]
    - persistence_base["previous_position"]
    == 1
)

# Both categories must be available
persistence_base["valid_transition"] = (
    persistence_base["is_adjacent"]
    & persistence_base["level_2"].notna()
    & persistence_base["previous_category"].notna()
)

# Retain valid adjacent transitions
transition_df = persistence_base[
    persistence_base["valid_transition"]
].copy()

# Identify repeated categories
transition_df["is_repeated"] = (
    transition_df["level_2"]
    == transition_df["previous_category"]
).astype(int)

print("=== VALID BEHAVIORAL TRANSITIONS ===")
print("Total historical interactions:", len(persistence_base))
print("Potential transitions:",
      len(persistence_base) - persistence_base["user_id"].nunique())
print("Valid transitions:", len(transition_df))
print("Users with valid transitions:",
      transition_df["user_id"].nunique())

display(
    transition_df[
        [
            "user_id",
            "position",
            "previous_category",
            "level_2",
            "is_repeated"
        ]
    ].head(10)
)

In [ ]:
# Step 28.2: Calculate user-level persistence

persistence_df = (
    transition_df
    .groupby("user_id")
    .agg(
        valid_transitions=("is_repeated", "size"),
        repeated_transitions=("is_repeated", "sum")
    )
    .reset_index()
)

# Calculate sequential repetition rate
persistence_df["repetition_rate"] = (
    persistence_df["repeated_transitions"]
    / persistence_df["valid_transitions"]
)

# Retain all eligible users
persistence_df = measurement_df[["user_id"]].merge(
    persistence_df,
    on="user_id",
    how="left",
    validate="one_to_one"
)

# Users without valid transitions remain undefined
persistence_df["valid_transitions"] = (
    persistence_df["valid_transitions"].fillna(0).astype(int)
)

persistence_df["repeated_transitions"] = (
    persistence_df["repeated_transitions"].fillna(0).astype(int)
)

print("=== CONSUMER PERSISTENCE DATASET ===")
print("Shape:", persistence_df.shape)

display(persistence_df.head(10))

In [ ]:
# Step 28.3: Validate and explore persistence

import numpy as np
import matplotlib.pyplot as plt

print("=== DATA QUALITY CHECK ===")
print("Total users:", len(persistence_df))
print("Unique users:", persistence_df["user_id"].nunique())
print("Duplicate users:",
      persistence_df["user_id"].duplicated().sum())

print("\nMissing values:")
print(persistence_df.isna().sum())

# Validation checks
assert len(persistence_df) == len(measurement_df)
assert persistence_df["user_id"].is_unique

assert (
    persistence_df["repeated_transitions"]
    <= persistence_df["valid_transitions"]
).all()

assert persistence_df["repetition_rate"].dropna().between(
    0, 1
).all()

print("\nAll validation checks: PASSED")

# Descriptive statistics
print("\n=== DESCRIPTIVE STATISTICS ===")

display(
    persistence_df.drop(columns="user_id")
    .describe()
    .round(4)
)

# Boundary values
print("\n=== PERSISTENCE PATTERNS ===")

print("Users with zero repetition:",
      (persistence_df["repetition_rate"] == 0).sum())

print("Users with complete repetition:",
      (persistence_df["repetition_rate"] == 1).sum())

print("Users without valid transitions:",
      persistence_df["repetition_rate"].isna().sum())

# Merge with previous measurements
persistence_analysis_df = diversity_analysis_df.merge(
    persistence_df,
    on="user_id",
    how="left",
    validate="one_to_one"
)

# Correlation analysis
correlation_columns = [
    "history_length",
    "category_count",
    "category_entropy",
    "normalized_entropy",
    "raw_tvd",
    "excess_tvd",
    "repetition_rate"
]

print("\n=== SPEARMAN CORRELATION ===")

display(
    persistence_analysis_df[correlation_columns]
    .corr(method="spearman")
    .round(3)
)

In [ ]:
# Step 28.4: Visualize behavioral persistence

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Figure A: Distribution of repetition rate
axes[0].hist(
    persistence_df["repetition_rate"].dropna(),
    bins=np.linspace(0, 1, 21),
    color="steelblue",
    edgecolor="white",
    alpha=0.85
)

axes[0].axvline(
    persistence_df["repetition_rate"].median(),
    color="darkorange",
    linestyle="--",
    label="Median"
)

axes[0].set_title("A. Behavioral Persistence")
axes[0].set_xlabel("Sequential Repetition Rate")
axes[0].set_ylabel("Number of Users")
axes[0].legend()

# Figure B: Diversity vs persistence
axes[1].scatter(
    persistence_analysis_df["category_entropy"],
    persistence_analysis_df["repetition_rate"],
    alpha=0.25,
    s=15,
    color="steelblue"
)

rho_entropy = persistence_analysis_df[
    ["category_entropy", "repetition_rate"]
].corr(method="spearman").iloc[0, 1]

axes[1].set_title(
    f"B. Diversity vs Persistence (ρ = {rho_entropy:.3f})"
)

axes[1].set_xlabel("Shannon Entropy")
axes[1].set_ylabel("Repetition Rate")

# Figure C: Temporal shift vs persistence
axes[2].scatter(
    persistence_analysis_df["excess_tvd"],
    persistence_analysis_df["repetition_rate"],
    alpha=0.25,
    s=15,
    color="steelblue"
)

rho_shift = persistence_analysis_df[
    ["excess_tvd", "repetition_rate"]
].corr(method="spearman").iloc[0, 1]

axes[2].set_title(
    f"C. Shift vs Persistence (ρ = {rho_shift:.3f})"
)

axes[2].set_xlabel("Excess TVD")
axes[2].set_ylabel("Repetition Rate")

plt.suptitle(
    "Consumer Behavioral Persistence and Its Relationships",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
# Step 29: Measure Consumer Exploration Behavior

def calculate_exploration(group):
    group = group.sort_values(["date", "item_id"])

    # Split before removing missing categories
    midpoint = len(group) // 2

    early = group.iloc[:midpoint]["level_2"].dropna()
    recent = group.iloc[midpoint:]["level_2"].dropna()

    early_categories = set(early)
    novel_mask = ~recent.isin(early_categories)

    return {
        "user_id": group["user_id"].iloc[0],
        "exploration_early_valid": len(early),
        "exploration_recent_valid": len(recent),
        "novel_interactions": int(novel_mask.sum()),
        "novel_category_count": len(
            set(recent) - early_categories
        ),
        "exploration_rate": novel_mask.mean()
    }

exploration_df = pd.DataFrame([
    calculate_exploration(group)
    for _, group in behavior_df.groupby("user_id")
])

# Validation
check = measurement_df.merge(
    exploration_df,
    on="user_id",
    validate="one_to_one"
)

assert len(exploration_df) == 1420
assert exploration_df["user_id"].is_unique
assert exploration_df.isna().sum().sum() == 0

assert (
    check["early_valid"]
    == check["exploration_early_valid"]
).all()

assert (
    check["recent_valid"]
    == check["exploration_recent_valid"]
).all()

assert exploration_df["exploration_rate"].between(0, 1).all()

# Exploration rate cannot exceed Raw TVD
assert (
    check["exploration_rate"]
    <= check["raw_tvd"] + 1e-10
).all()

print("=== DATA QUALITY CHECK ===")
print("Shape:", exploration_df.shape)
print("All validation checks: PASSED")

display(exploration_df.head(10))

print("\n=== DESCRIPTIVE STATISTICS ===")
display(
    exploration_df.drop(columns="user_id")
    .describe().round(4)
)

# Merge with existing measurements
exploration_analysis_df = persistence_analysis_df.merge(
    exploration_df,
    on="user_id",
    validate="one_to_one"
)

columns = [
    "history_length",
    "category_entropy",
    "repetition_rate",
    "raw_tvd",
    "excess_tvd",
    "exploration_rate"
]

print("\n=== SPEARMAN CORRELATION ===")
display(
    exploration_analysis_df[columns]
    .corr(method="spearman").round(3)
)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(
    exploration_df["exploration_rate"],
    bins=np.linspace(0, 1, 21),
    color="steelblue",
    edgecolor="white"
)
axes[0].set_title("Exploration Rate Distribution")
axes[0].set_xlabel("Exploration Rate")
axes[0].set_ylabel("Number of Users")

for ax, x in zip(axes[1:], ["raw_tvd", "excess_tvd"]):
    rho = exploration_analysis_df[
        [x, "exploration_rate"]
    ].corr(method="spearman").iloc[0, 1]

    ax.scatter(
        exploration_analysis_df[x],
        exploration_analysis_df["exploration_rate"],
        alpha=0.25,
        s=15
    )
    ax.set_title(f"{x} vs Exploration (ρ = {rho:.3f})")
    ax.set_xlabel(x)
    ax.set_ylabel("Exploration Rate")

plt.tight_layout()
plt.show()

In [ ]:
# Step 30: Excess Repetition Rate

def expected_repetition(sequence):
    counts = pd.Series(sequence).value_counts()
    n = len(sequence)

    return (
        (counts * (counts - 1)).sum()
        / (n * (n - 1))
    )

# Calculate exact null expectation
null_repetition_df = category_sequences[["user_id"]].copy()

null_repetition_df["null_repetition_rate"] = (
    category_sequences["category_sequence"]
    .apply(expected_repetition)
)

# Merge with observed repetition
excess_repetition_df = persistence_df.merge(
    null_repetition_df,
    on="user_id",
    validate="one_to_one"
)

excess_repetition_df["excess_repetition_rate"] = (
    excess_repetition_df["repetition_rate"]
    - excess_repetition_df["null_repetition_rate"]
)

# Validation
assert len(excess_repetition_df) == 1420
assert excess_repetition_df["user_id"].is_unique
assert excess_repetition_df.isna().sum().sum() == 0

assert excess_repetition_df[
    "null_repetition_rate"
].between(0, 1).all()

assert excess_repetition_df[
    "excess_repetition_rate"
].between(-1, 1).all()

print("=== DATA QUALITY CHECK ===")
print("All validation checks: PASSED")

display(excess_repetition_df.head(10))

metrics = [
    "repetition_rate",
    "null_repetition_rate",
    "excess_repetition_rate"
]

print("\n=== DESCRIPTIVE STATISTICS ===")
display(
    excess_repetition_df[metrics]
    .describe().round(4)
)

# Merge with existing behavioral features
excess_repetition_analysis_df = (
    exploration_analysis_df.merge(
        excess_repetition_df[
            [
                "user_id",
                "null_repetition_rate",
                "excess_repetition_rate"
            ]
        ],
        on="user_id",
        validate="one_to_one"
    )
)

columns = [
    "history_length",
    "category_entropy",
    "raw_tvd",
    "excess_tvd",
    "exploration_rate",
    "repetition_rate",
    "null_repetition_rate",
    "excess_repetition_rate"
]

print("\n=== SPEARMAN CORRELATION ===")
display(
    excess_repetition_analysis_df[columns]
    .corr(method="spearman").round(3)
)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(
    excess_repetition_df["excess_repetition_rate"],
    bins=30,
    color="steelblue",
    edgecolor="white"
)
axes[0].axvline(0, color="black", linestyle="--")
axes[0].set_title("Excess Repetition Distribution")
axes[0].set_xlabel("Excess Repetition Rate")
axes[0].set_ylabel("Number of Users")

for ax, y in zip(
    axes[1:],
    ["repetition_rate", "excess_repetition_rate"]
):
    rho = excess_repetition_analysis_df[
        ["category_entropy", y]
    ].corr(method="spearman").iloc[0, 1]

    ax.scatter(
        excess_repetition_analysis_df["category_entropy"],
        excess_repetition_analysis_df[y],
        alpha=0.25,
        s=15
    )

    ax.set_title(f"Entropy vs {y}\nρ = {rho:.3f}")
    ax.set_xlabel("Shannon Entropy")
    ax.set_ylabel(y)

plt.tight_layout()
plt.show()

In [ ]:
# Step 31: Construct Final Behavioral Feature Dataset

feature_df = excess_repetition_analysis_df.copy()

# Log-transform history length
feature_df["log_history_length"] = np.log1p(
    feature_df["history_length"]
)

primary_features = [
    "log_history_length",
    "category_entropy",
    "excess_tvd",
    "excess_repetition_rate",
    "exploration_rate"
]

comparison_features = [
    "history_length",
    "valid_history_length",
    "category_count",
    "normalized_entropy",
    "raw_tvd",
    "null_mean",
    "repetition_rate",
    "null_repetition_rate"
]

final_feature_df = feature_df[
    ["user_id"] + primary_features + comparison_features
].copy()

# Validation
assert len(final_feature_df) == 1420
assert final_feature_df["user_id"].is_unique
assert final_feature_df.isna().sum().sum() == 0
assert np.isfinite(
    final_feature_df.drop(columns="user_id")
).all().all()

print("=== FINAL BEHAVIORAL FEATURE DATASET ===")
print("Shape:", final_feature_df.shape)
print("Users:", final_feature_df["user_id"].nunique())
print("All validation checks: PASSED")

print("\n=== PRIMARY FEATURES ===")
display(
    final_feature_df[["user_id"] + primary_features].head(10)
)

print("\n=== DESCRIPTIVE STATISTICS ===")
display(
    final_feature_df[primary_features].describe().round(4)
)

# Save dataset
output_path = "/content/amazon_behavioral_features.csv"

final_feature_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

In [ ]:
# Step 32: Joint Measurement Analysis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    1, 2, figsize=(13, 5),
    gridspec_kw={"width_ratios": [1, 1.3]}
)

# Panel A: Before vs. after correction
pairs = [
    ("raw_tvd", "excess_tvd"),
    ("repetition_rate", "excess_repetition_rate")
]

before, after = [], []

for raw, corrected in pairs:
    before.append(
        final_feature_df["category_entropy"]
        .corr(final_feature_df[raw], method="spearman")
    )
    after.append(
        final_feature_df["category_entropy"]
        .corr(final_feature_df[corrected], method="spearman")
    )

x = np.arange(2)

bars1 = axes[0].bar(
    x - 0.18, before, 0.36,
    label="Raw", color="steelblue"
)

bars2 = axes[0].bar(
    x + 0.18, after, 0.36,
    label="Corrected", color="seagreen"
)

axes[0].bar_label(bars1, fmt="%.3f", padding=3)
axes[0].bar_label(bars2, fmt="%.3f", padding=3)

axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xticks(x, ["TVD", "Repetition"])
axes[0].set_ylim(-1, 1)
axes[0].set_ylabel("Spearman Correlation with Entropy")
axes[0].set_title("A. Before vs. After Correction")
axes[0].legend()

# Panel B: Primary feature correlations
corr = final_feature_df[primary_features].corr(
    method="spearman"
)

labels = [
    "History Length",
    "Diversity",
    "Temporal Shift",
    "Persistence",
    "Exploration"
]

im = axes[1].imshow(
    corr, cmap="RdBu_r", vmin=-1, vmax=1
)

axes[1].set_xticks(
    range(5), labels, rotation=35, ha="right"
)
axes[1].set_yticks(range(5), labels)

for i in range(5):
    for j in range(5):
        value = corr.iloc[i, j]
        axes[1].text(
            j, i, f"{value:.2f}",
            ha="center", va="center",
            color="white" if abs(value) > 0.6 else "black"
        )

axes[1].set_title("B. Primary Feature Correlations")
fig.colorbar(im, ax=axes[1], shrink=0.8)

plt.suptitle(
    "Consumer Behavioral Measurement and Feature Relationships",
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    "/content/behavioral_measurement_summary.png",
    dpi=300, bbox_inches="tight"
)

plt.show()

print("=== PRIMARY FEATURE CORRELATIONS ===")
display(corr.round(3))

In [ ]:
# Step 33: Robustness Checks

# 1. History length sensitivity
robustness_df = final_feature_df.copy()

robustness_df["length_group"] = pd.cut(
    robustness_df["history_length"],
    bins=[0, 8, 12, np.inf],
    labels=["6–8", "9–12", "13+"]
)

results = []

for name, group in robustness_df.groupby(
    "length_group", observed=True
):
    row = {
        "History Group": name,
        "Users": len(group)
    }

    for col in [
        "raw_tvd",
        "excess_tvd",
        "repetition_rate",
        "excess_repetition_rate"
    ]:
        row[col] = group["category_entropy"].corr(
            group[col], method="spearman"
        )

    results.append(row)

robustness_results = pd.DataFrame(results)

print("=== HISTORY LENGTH SENSITIVITY ===")
display(robustness_results.round(4))

# 2. User-level bootstrap confidence intervals
rng = np.random.default_rng(2027)
B = 5000

bootstrap_results = []

for col in ["excess_tvd", "excess_repetition_rate"]:
    values = final_feature_df[col].to_numpy()

    bootstrap_means = rng.choice(
        values,
        size=(B, len(values)),
        replace=True
    ).mean(axis=1)

    lower, upper = np.percentile(
        bootstrap_means, [2.5, 97.5]
    )

    bootstrap_results.append({
        "Metric": col,
        "Mean": values.mean(),
        "95% CI Lower": lower,
        "95% CI Upper": upper
    })

print("\n=== BOOTSTRAP CONFIDENCE INTERVALS ===")
display(pd.DataFrame(bootstrap_results).round(4))

In [ ]:
# Step 35.1: Check recommendation experiment resources

import os

print("=== AVAILABLE FILES ===")

for file in os.listdir("/content"):
    if file.endswith((".csv", ".json", ".jsonl")):
        print(file)

print("\n=== AVAILABLE VARIABLES ===")

for name in [
    "final_feature_df",
    "recommendation_tasks",
    "expanded_history",
    "metadata_df",
    "recommendation_prompt",
    "client"
]:
    print(f"{name}: {name in globals()}")

In [ ]:
# Step 35.2: Reconstruct candidate pools

import pandas as pd
import numpy as np
import ast

meta = metadata_df.copy()
meta["item_id"] = meta["item_id"].astype(str)
meta = meta.drop_duplicates("item_id")

def parse_cat(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            value = ast.literal_eval(x)
            return value if isinstance(value, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

meta["category_path"] = meta["categories"].apply(parse_cat)

meta["level_1"] = meta["category_path"].apply(
    lambda x: x[0] if len(x) >= 1 else None
)

meta["level_2"] = meta["category_path"].apply(
    lambda x: x[1] if len(x) >= 2 else None
)

eligible = set(final_feature_df["user_id"].astype(str))

targets = expanded_targets.copy()
targets["user_id"] = targets["user_id"].astype(str)
targets["item_id"] = targets["item_id"].astype(str)

targets = targets[
    targets["user_id"].isin(eligible)
].merge(
    meta[["item_id", "level_1", "level_2"]],
    on="item_id",
    how="left",
    validate="many_to_one"
)

print("=== TARGET VALIDATION ===")
print("Eligible users:", len(eligible))
print("Target records:", len(targets))
print("Unique target users:", targets["user_id"].nunique())
print("Duplicate users:", targets["user_id"].duplicated().sum())
print("Missing target L1:", targets["level_1"].isna().sum())
print("Missing target L2:", targets["level_2"].isna().sum())
print("Available products:", len(meta))

assert len(targets) == len(eligible)
assert targets["user_id"].is_unique
assert set(targets["user_id"]) == eligible

display(targets.head())

In [ ]:
# Step 35.3: Generate reproducible candidate sets

import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)

prediction_targets = targets.dropna(
    subset=["level_1", "level_2"]
).sort_values("user_id").copy()

meta_pool = meta.drop_duplicates("item_id").copy()
all_items = set(meta_pool["item_id"])

l1_pools = {
    k: set(g["item_id"])
    for k, g in meta_pool.dropna(
        subset=["level_1"]
    ).groupby("level_1")
}

l2_pools = {
    k: set(g["item_id"])
    for k, g in meta_pool.dropna(
        subset=["level_1", "level_2"]
    ).groupby(["level_1", "level_2"])
}

history_map = (
    expanded_history
    .assign(
        user_id=lambda x: x["user_id"].astype(str),
        item_id=lambda x: x["item_id"].astype(str)
    )
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

tasks = []

for _, row in prediction_targets.iterrows():

    user = str(row["user_id"])
    target = str(row["item_id"])
    l1, l2 = row["level_1"], row["level_2"]

    excluded = history_map.get(user, set()) | {target}

    same_l2 = l2_pools.get((l1, l2), set()) - excluded
    same_l1 = l1_pools.get(l1, set()) - excluded - same_l2

    selected = []

    def sample_from(pool, n):
        available = sorted(set(pool) - set(selected))
        k = min(n, len(available))
        if k == 0:
            return []
        return rng.choice(
            available, size=k, replace=False
        ).tolist()

    selected += sample_from(same_l2, 9)
    selected += sample_from(same_l1, 10)

    if len(selected) < 19:
        selected += sample_from(
            l1_pools.get(l1, set()) - excluded,
            19 - len(selected)
        )

    if len(selected) < 19:
        selected += sample_from(
            all_items - excluded,
            19 - len(selected)
        )

    assert len(selected) == 19

    candidates = selected + [target]
    rng.shuffle(candidates)

    tasks.append({
        "user_id": user,
        "groundtruth": target,
        "candidate_list": candidates,
        "target_level_1": l1,
        "target_level_2": l2,
        "repeat_target": target in history_map.get(user, set()),
        "same_l2_negatives": sum(x in same_l2 for x in selected),
        "same_l1_negatives": sum(
            x in l1_pools.get(l1, set()) for x in selected
        )
    })

recommendation_tasks = pd.DataFrame(tasks)

# Quality checks
assert recommendation_tasks["user_id"].is_unique

assert recommendation_tasks["candidate_list"].apply(
    lambda x: len(x) == len(set(x)) == 20
).all()

assert recommendation_tasks.apply(
    lambda r: r["groundtruth"] in r["candidate_list"],
    axis=1
).all()

print("=== CANDIDATE GENERATION ===")
print("Total tasks:", len(recommendation_tasks))
print("All quality checks passed.")

print("\n=== CANDIDATE DIFFICULTY ===")
display(recommendation_tasks[
    ["same_l2_negatives", "same_l1_negatives"]
].describe().round(4))

print("\n=== REPEAT TARGETS ===")
print(recommendation_tasks["repeat_target"].value_counts())

recommendation_tasks.to_json(
    "/content/amazon_recommendation_tasks.json",
    orient="records",
    indent=2
)

print("\nCandidate tasks saved.")

In [ ]:
# Step 35.4: Construct recommendation prompts

import ast

history = expanded_history.copy()
history["user_id"] = history["user_id"].astype(str)
history["item_id"] = history["item_id"].astype(str)

meta_lookup = meta.set_index("item_id")

def format_product(item_id):
    row = meta_lookup.loc[str(item_id)]
    title = str(row["title"])[:140]
    category = " > ".join(
        str(c) for c in row["category_path"][:3]
    )

    return (
        f"{item_id} | {title} | "
        f"Category: {category}"
    )

def build_prompt(row):
    uid = str(row["user_id"])
    candidates = list(row["candidate_list"])

    user_hist = (
        history[history["user_id"] == uid]
        .sort_values(["date", "item_id"])
    )

    history_text = "\n".join(
        f"{i+1}. {format_product(item)}"
        for i, item in enumerate(user_hist["item_id"])
    )

    candidate_text = "\n".join(
        f"{i+1}. {format_product(item)}"
        for i, item in enumerate(candidates)
    )

    return f"""
You are an Amazon product recommendation assistant.

TASK:
Rank the 20 candidate products according to how likely
the user is to select them next.

USER HISTORY (oldest to most recent):
{history_text}

CANDIDATE PRODUCTS:
{candidate_text}

INSTRUCTIONS:
- Infer user preferences from the purchase history.
- Consider both long-term and recent preferences.
- Rank all 20 candidates from most to least likely.
- Use only the provided information.
- Return valid JSON in the following format:
  {{"ranking": ["item_id_1", "item_id_2", "..."]}}
- Include exactly 20 unique candidate IDs.
- Do not provide explanations.
"""

# Construct and validate all prompts
recommendation_tasks["prompt"] = (
    recommendation_tasks.apply(build_prompt, axis=1)
)

recommendation_tasks["prompt_length"] = (
    recommendation_tasks["prompt"].str.len()
)

recommendation_tasks["estimated_tokens"] = (
    recommendation_tasks["prompt_length"] / 4
).round().astype(int)

assert recommendation_tasks["prompt"].notna().all()

print("=== PROMPT CONSTRUCTION ===")
print("Total prompts:", len(recommendation_tasks))

print("\n=== PROMPT SIZE ===")
display(recommendation_tasks[
    ["prompt_length", "estimated_tokens"]
].describe().round(2))

print("\n=== EXAMPLE PROMPT ===")
print(recommendation_tasks.iloc[0]["prompt"])

In [ ]:
# Step 35.5: Extended Gemini Pilot — Resume Version

import json
import time
import hashlib
import pandas as pd

from pathlib import Path
from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata

MODEL = "gemini-3.1-flash-lite"
OUTPUT = Path("/content/gemini_attempt3_results.json")

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

# 1. Reproducible random order
ordered_tasks = recommendation_tasks.sample(
    frac=1, random_state=2026
).reset_index(drop=True)

N = 100
pilot_tasks = ordered_tasks.head(N)

# 2. Load existing results
results = (
    json.loads(OUTPUT.read_text())
    if OUTPUT.exists() else []
)

result_lookup = {
    r["user_id"]: r for r in results
}

# 3. Prioritize unattempted users
pending_tasks = pilot_tasks[
    ~pilot_tasks["user_id"].astype(str).isin(
        result_lookup.keys()
    )
].copy()

print("=== RESUME STATUS ===")
print("Planned users:", N)
print("Previously attempted:", N - len(pending_tasks))
print("Unattempted users:", len(pending_tasks))

# 4. Generate rankings
for i, (_, row) in enumerate(pending_tasks.iterrows(), 1):

    uid = str(row["user_id"])
    candidates = list(row["candidate_list"])
    target = str(row["groundtruth"])
    prompt = row["prompt"]

    prompt_hash = hashlib.sha256(
        prompt.encode("utf-8")
    ).hexdigest()

    record = {
        "user_id": uid,
        "model": MODEL,
        "prompt_sha256": prompt_hash,
        "groundtruth": target,
        "valid": False,
        "ranking": None,
        "target_rank": None,
        "hit_at_5": None,
        "failure_at_5": None,
        "input_tokens": None,
        "output_tokens": None,
        "error": None
    }

    stop_batch = False

    for attempt in range(3):

        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0
                )
            )

            ranking = json.loads(response.text)["ranking"]

            valid = (
                isinstance(ranking, list)
                and len(ranking) == 20
                and len(set(ranking)) == 20
                and set(ranking) == set(candidates)
            )

            if not valid:
                raise ValueError("Invalid ranking")

            rank = ranking.index(target) + 1
            usage = response.usage_metadata

            record.update({
                "valid": True,
                "ranking": ranking,
                "target_rank": rank,
                "hit_at_5": int(rank <= 5),
                "failure_at_5": int(rank > 5),
                "input_tokens": (
                    usage.prompt_token_count
                    if usage else None
                ),
                "output_tokens": (
                    usage.candidates_token_count
                    if usage else None
                ),
                "error": None
            })

            break

        except APIError as e:

            record["error"] = f"API error {e.code}"

            if e.code in (429, 500, 502, 503, 504):

                if attempt < 2:
                    wait = 30 * 2**attempt
                    print(f"API error {e.code}. Retry in {wait}s.")
                    time.sleep(wait)
                else:
                    stop_batch = True

            else:
                stop_batch = True
                break

        except (ValueError, KeyError, TypeError) as e:
            record["error"] = str(e)

    result_lookup[uid] = record

    # 5. Save after every user
    temp = OUTPUT.with_suffix(".tmp")

    temp.write_text(
        json.dumps(
            list(result_lookup.values()),
            indent=2,
            ensure_ascii=False
        )
    )

    temp.replace(OUTPUT)

    print(
        f"[{i}/{len(pending_tasks)}] "
        f"Valid={record['valid']} | "
        f"Rank={record['target_rank']} | "
        f"Failure@5={record['failure_at_5']} | "
        f"Error={record['error']}"
    )

    if stop_batch:
        print("Batch paused. Check API status or quota.")
        break

    time.sleep(3)

# 6. Extended pilot summary
pilot_ids = set(pilot_tasks["user_id"].astype(str))

df_pilot = pd.DataFrame([
    r for uid, r in result_lookup.items()
    if uid in pilot_ids
])

print("\n=== EXTENDED PILOT SUMMARY ===")
print("Planned:", N)
print("Attempted:", len(df_pilot))
print("Unattempted:", N - len(df_pilot))

if not df_pilot.empty:

    valid_df = df_pilot[df_pilot["valid"]].copy()
    invalid_df = df_pilot[~df_pilot["valid"]].copy()

    print("Valid rankings:", len(valid_df))
    print("Invalid rankings:", len(invalid_df))
    print("Validity rate:", round(len(valid_df) / len(df_pilot), 4))

    if len(valid_df):
        print("Hit@5:", round(valid_df["hit_at_5"].mean(), 4))
        print("Failure@5:", round(valid_df["failure_at_5"].mean(), 4))
        print("Mean input tokens:", valid_df["input_tokens"].mean())
        print("Mean output tokens:", valid_df["output_tokens"].mean())

    if len(invalid_df):
        print("\n=== ERROR BREAKDOWN ===")
        print(invalid_df["error"].value_counts())

print("\nSaved:", OUTPUT)

In [ ]:
from google.colab import files

files.download("/content/gemini_attempt3_results.json")
files.download("/content/amazon_recommendation_tasks.json")

In [ ]:
# Step 36.1: Construct prediction dataset

import pandas as pd
import numpy as np
import json

# 1. Load behavioral features
features = final_feature_df.copy()
features["user_id"] = features["user_id"].astype(str)

# 2. Load recommendation labels
with open("/content/gemini_attempt3_results.json") as f:
    label_df = pd.DataFrame(json.load(f))

label_df["user_id"] = label_df["user_id"].astype(str)

# 3. Keep valid rankings only
valid_labels = label_df[
    label_df["valid"] == True
].copy()

valid_labels["failure_at_5"] = (
    valid_labels["failure_at_5"].astype(int)
)

# 4. Merge X and Y
prediction_df = features.merge(
    valid_labels[
        ["user_id", "target_rank", "failure_at_5"]
    ],
    on="user_id",
    how="inner",
    validate="one_to_one"
)

primary_features = [
    "log_history_length",
    "category_entropy",
    "excess_tvd",
    "excess_repetition_rate",
    "exploration_rate"
]

X = prediction_df[primary_features].copy()
y = prediction_df["failure_at_5"].copy()

# 5. Quality checks
assert len(prediction_df) == len(valid_labels)
assert prediction_df["user_id"].is_unique
assert X.notna().all().all()
assert np.isfinite(X.to_numpy()).all()
assert set(y.unique()) == {0, 1}
assert (
    y == (prediction_df["target_rank"] > 5).astype(int)
).all()

print("=== PREDICTION DATASET ===")
print("Total users:", len(prediction_df))
print("X shape:", X.shape)
print("Y shape:", y.shape)
print("Missing values:", X.isna().sum().sum())

print("\n=== LABEL DISTRIBUTION ===")
print(y.value_counts().sort_index())
print("Failure rate:", round(y.mean(), 4))

print("\n=== FEATURE SUMMARY ===")
display(X.describe().round(4))

# 6. Compare pilot sample with full measurement sample
comparison = pd.DataFrame({
    "Full Sample Mean": features[primary_features].mean(),
    "Pilot Sample Mean": X.mean(),
    "Full Sample SD": features[primary_features].std()
})

comparison["Standardized Mean Difference"] = (
    comparison["Pilot Sample Mean"]
    - comparison["Full Sample Mean"]
) / comparison["Full Sample SD"]

print("\n=== SAMPLE REPRESENTATIVENESS ===")
display(comparison.round(4))

# 7. Save prediction dataset
prediction_df.to_csv(
    "/content/amazon_prediction_dataset.csv",
    index=False
)

print("\nAll quality checks passed.")

In [ ]:
# Step 36.2: Model Specification

model_df = prediction_df.copy()

# 1. Interaction terms
model_df["shift_x_persistence"] = (
    model_df["excess_tvd"]
    * model_df["excess_repetition_rate"]
)

model_df["shift_x_exploration"] = (
    model_df["excess_tvd"]
    * model_df["exploration_rate"]
)

# 2. Feature specifications
model_features = {
    "M0_Baseline": [
        "log_history_length",
        "category_entropy"
    ],

    "M1_Raw_Shift": [
        "log_history_length",
        "category_entropy",
        "raw_tvd"
    ],

    "M2_Corrected_Shift": [
        "log_history_length",
        "category_entropy",
        "excess_tvd"
    ],

    "M3_Behavioral_Heterogeneity": [
        "log_history_length",
        "category_entropy",
        "excess_tvd",
        "excess_repetition_rate",
        "exploration_rate"
    ],

    "M4_Interactions": [
        "log_history_length",
        "category_entropy",
        "excess_tvd",
        "excess_repetition_rate",
        "exploration_rate",
        "shift_x_persistence",
        "shift_x_exploration"
    ]
}

# 3. Prepare model inputs
X_models = {
    name: model_df[cols].copy()
    for name, cols in model_features.items()
}

y = model_df["failure_at_5"].astype(int)

# 4. Quality checks
for name, X_model in X_models.items():

    assert len(X_model) == len(y)
    assert X_model.notna().all().all()
    assert np.isfinite(X_model.to_numpy()).all()

    print(f"{name}: {X_model.shape}")

print("\n=== INTERACTION TERMS ===")

display(model_df[
    ["shift_x_persistence", "shift_x_exploration"]
].describe().round(4))

print("\nAll model specifications validated.")

In [ ]:
# Step 36.3: Regularized Logistic Regression

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    brier_score_loss,
    accuracy_score
)
from sklearn.dummy import DummyClassifier

# 1. Fixed cross-validation splits
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

cv_splits = list(cv.split(X_models["M0_Baseline"], y))

# 2. Store evaluation results
lr_results = []
lr_oof_predictions = {}

def evaluate_predictions(name, y_true, probabilities):

    return {
        "Model": name,
        "Log Loss": log_loss(y_true, probabilities),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "Brier Score": brier_score_loss(y_true, probabilities),
        "Accuracy": accuracy_score(
            y_true, (probabilities >= 0.5).astype(int)
        )
    }

# 3. Naive probability baseline
dummy = DummyClassifier(strategy="prior")

dummy_probs = cross_val_predict(
    dummy,
    X_models["M0_Baseline"],
    y,
    cv=cv_splits,
    method="predict_proba"
)[:, 1]

lr_oof_predictions["Naive_Baseline"] = dummy_probs

lr_results.append(
    evaluate_predictions("Naive_Baseline", y, dummy_probs)
)

# 4. Train M0–M4
for name, X_model in X_models.items():

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
            random_state=2026
        ))
    ])

    probabilities = cross_val_predict(
        pipeline,
        X_model,
        y,
        cv=cv_splits,
        method="predict_proba"
    )[:, 1]

    lr_oof_predictions[name] = probabilities

    lr_results.append(
        evaluate_predictions(name, y, probabilities)
    )

# 5. Evaluation table
lr_results_df = pd.DataFrame(lr_results)

print("=== LOGISTIC REGRESSION RESULTS ===")
display(lr_results_df.round(4))

# 6. Incremental predictive value
loss_map = lr_results_df.set_index("Model")["Log Loss"]

comparisons = [
    ("Naive_Baseline", "M0_Baseline"),
    ("M0_Baseline", "M1_Raw_Shift"),
    ("M0_Baseline", "M2_Corrected_Shift"),
    ("M1_Raw_Shift", "M2_Corrected_Shift"),
    ("M2_Corrected_Shift", "M3_Behavioral_Heterogeneity"),
    ("M3_Behavioral_Heterogeneity", "M4_Interactions")
]

incremental_results = []

for base, extended in comparisons:

    improvement = loss_map[base] - loss_map[extended]

    incremental_results.append({
        "Comparison": f"{base} → {extended}",
        "Log Loss Improvement": improvement,
        "Relative Improvement (%)":
            improvement / loss_map[base] * 100
    })

lr_incremental_df = pd.DataFrame(incremental_results)

print("\n=== INCREMENTAL PREDICTIVE VALUE ===")
display(lr_incremental_df.round(4))

# 7. Save out-of-fold predictions
lr_oof_df = model_df[
    ["user_id", "failure_at_5"]
].copy()

for name, probabilities in lr_oof_predictions.items():
    lr_oof_df[name] = probabilities

lr_oof_df.to_csv(
    "/content/logistic_oof_predictions.csv",
    index=False
)

print("\nAll logistic regression experiments completed.")

In [ ]:
# Step 37: Random Forest Classification

from sklearn.ensemble import RandomForestClassifier

rf_results = []
rf_oof_predictions = {}

# Same naive baseline
rf_results.append(
    evaluate_predictions("Naive_Baseline", y, dummy_probs)
)

rf_oof_predictions["Naive_Baseline"] = dummy_probs

# Train M0–M3
for name, X_model in X_models.items():

    if name == "M4_Interactions":
        continue

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=3,
        min_samples_leaf=5,
        max_features=None,
        random_state=2026,
        n_jobs=-1
    )

    probabilities = cross_val_predict(
        rf,
        X_model,
        y,
        cv=cv_splits,
        method="predict_proba"
    )[:, 1]

    rf_oof_predictions[name] = probabilities

    rf_results.append(
        evaluate_predictions(name, y, probabilities)
    )

# Evaluation results
rf_results_df = pd.DataFrame(rf_results)

print("=== RANDOM FOREST RESULTS ===")
display(rf_results_df.round(4))

# Compare Logistic Regression and Random Forest
comparison_df = lr_results_df.merge(
    rf_results_df,
    on="Model",
    suffixes=("_LR", "_RF")
)

comparison_df["RF Log Loss Improvement"] = (
    comparison_df["Log Loss_LR"]
    - comparison_df["Log Loss_RF"]
)

print("\n=== LR VS RANDOM FOREST ===")

display(comparison_df[[
    "Model",
    "Log Loss_LR",
    "Log Loss_RF",
    "ROC-AUC_LR",
    "ROC-AUC_RF",
    "RF Log Loss Improvement"
]].round(4))

# Save out-of-fold predictions
rf_oof_df = model_df[
    ["user_id", "failure_at_5"]
].copy()

for name, probabilities in rf_oof_predictions.items():
    rf_oof_df[name] = probabilities

rf_oof_df.to_csv(
    "/content/random_forest_oof_predictions.csv",
    index=False
)

print("\nAll Random Forest experiments completed.")

In [ ]:
# Step 38.1: Repeated Cross-Validation

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import cross_validate

repeated_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=2026
)

repeated_splits = list(
    repeated_cv.split(X_models["M0_Baseline"], y)
)

robustness_records = []

scoring = {
    "log_loss": "neg_log_loss",
    "auc": "roc_auc",
    "brier": "neg_brier_score"
}

def repeated_evaluation(algorithm, name, estimator, X_input):

    scores = cross_validate(
        estimator,
        X_input,
        y,
        cv=repeated_splits,
        scoring=scoring,
        n_jobs=-1
    )

    for i in range(len(repeated_splits)):

        robustness_records.append({
            "Algorithm": algorithm,
            "Model": name,
            "Repeat": i // 5 + 1,
            "Fold": i % 5 + 1,
            "Log Loss": -scores["test_log_loss"][i],
            "ROC-AUC": scores["test_auc"][i],
            "Brier Score": -scores["test_brier"][i]
        })

# 1. Naive baseline
repeated_evaluation(
    "Naive",
    "Naive_Baseline",
    DummyClassifier(strategy="prior"),
    X_models["M0_Baseline"]
)

# 2. Logistic Regression
for name, X_model in X_models.items():

    lr = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
            random_state=2026
        ))
    ])

    repeated_evaluation(
        "Logistic Regression", name, lr, X_model
    )

# 3. Random Forest
for name, X_model in X_models.items():

    if name == "M4_Interactions":
        continue

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=3,
        min_samples_leaf=5,
        max_features=None,
        random_state=2026,
        n_jobs=1
    )

    repeated_evaluation(
        "Random Forest", name, rf, X_model
    )

# 4. Summarize repeated results
repeated_results_df = pd.DataFrame(robustness_records)

repeat_means = (
    repeated_results_df
    .groupby(["Algorithm", "Model", "Repeat"])[
        ["Log Loss", "ROC-AUC", "Brier Score"]
    ]
    .mean()
    .reset_index()
)

robustness_summary = (
    repeat_means
    .groupby(["Algorithm", "Model"])
    .agg(
        Mean_Log_Loss=("Log Loss", "mean"),
        SD_Log_Loss=("Log Loss", "std"),
        Mean_AUC=("ROC-AUC", "mean"),
        SD_AUC=("ROC-AUC", "std"),
        Mean_Brier=("Brier Score", "mean")
    )
    .reset_index()
    .sort_values("Mean_Log_Loss")
)

print("=== REPEATED CROSS-VALIDATION ===")
display(robustness_summary.round(4))

repeated_results_df.to_csv(
    "/content/repeated_cv_results.csv",
    index=False
)

print("\nTotal fold evaluations:", len(repeated_results_df))
print("All robustness experiments completed.")

In [ ]:
# Step 38.2: Probability Calibration

import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

calibration_models = {
    "Naive Baseline": dummy_probs,

    "LR M0": lr_oof_predictions[
        "M0_Baseline"
    ],

    "LR M3": lr_oof_predictions[
        "M3_Behavioral_Heterogeneity"
    ],

    "RF M3": rf_oof_predictions[
        "M3_Behavioral_Heterogeneity"
    ]
}

fig, ax = plt.subplots(figsize=(7, 6))

ax.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="gray",
    label="Perfect Calibration"
)

calibration_records = []

for name, probs in calibration_models.items():

    observed, predicted = calibration_curve(
        y,
        probs,
        n_bins=4,
        strategy="uniform"
    )

    ax.plot(
        predicted,
        observed,
        marker="o",
        linewidth=2,
        label=name
    )

    calibration_records.append({
        "Model": name,
        "Mean Predicted Risk": np.mean(probs),
        "Observed Failure Rate": y.mean(),
        "Calibration Bias": np.mean(probs) - y.mean(),
        "Brier Score": brier_score_loss(y, probs)
    })

ax.set_xlabel("Mean Predicted Failure Probability")
ax.set_ylabel("Observed Failure Frequency")
ax.set_title("Calibration of Recommendation Failure Risk")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

plt.savefig(
    "/content/prediction_calibration.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

calibration_summary = pd.DataFrame(
    calibration_records
)

print("=== CALIBRATION SUMMARY ===")
display(calibration_summary.round(4))

In [ ]:
# Step 38.3: Final Model Evaluation Figure

import matplotlib.pyplot as plt
import numpy as np

model_order = [
    "M0_Baseline",
    "M1_Raw_Shift",
    "M2_Corrected_Shift",
    "M3_Behavioral_Heterogeneity",
    "M4_Interactions"
]

lr_plot = (
    robustness_summary[
        robustness_summary["Algorithm"] == "Logistic Regression"
    ]
    .set_index("Model")
    .reindex(model_order)
)

rf_plot = (
    robustness_summary[
        robustness_summary["Algorithm"] == "Random Forest"
    ]
    .set_index("Model")
    .reindex(model_order)
)

naive_loss = robustness_summary.loc[
    robustness_summary["Algorithm"] == "Naive",
    "Mean_Log_Loss"
].iloc[0]

x = np.arange(5)
width = 0.34

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: Log Loss
ax = axes[0]

ax.bar(
    x - width/2,
    lr_plot["Mean_Log_Loss"],
    width,
    yerr=lr_plot["SD_Log_Loss"],
    capsize=4,
    label="Logistic Regression",
    color="#527DA5"
)

ax.bar(
    x + width/2,
    rf_plot["Mean_Log_Loss"],
    width,
    yerr=rf_plot["SD_Log_Loss"],
    capsize=4,
    label="Random Forest",
    color="#D4A373"
)

ax.axhline(
    naive_loss,
    linestyle="--",
    color="black",
    label="Naive Baseline"
)

ax.set_title("A. Predictive Performance")
ax.set_ylabel("Mean Log Loss (Lower is Better)")
ax.set_xticks(x)
ax.set_xticklabels(["M0", "M1", "M2", "M3", "M4"])
ax.set_ylim(0, 0.85)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.2)

# Panel B: ROC-AUC
ax = axes[1]

ax.bar(
    x - width/2,
    lr_plot["Mean_AUC"],
    width,
    yerr=lr_plot["SD_AUC"],
    capsize=4,
    label="Logistic Regression",
    color="#527DA5"
)

ax.bar(
    x + width/2,
    rf_plot["Mean_AUC"],
    width,
    yerr=rf_plot["SD_AUC"],
    capsize=4,
    label="Random Forest",
    color="#D4A373"
)

ax.axhline(
    0.5,
    linestyle="--",
    color="black",
    label="Random Discrimination"
)

ax.set_title("B. Discrimination Performance")
ax.set_ylabel("Mean ROC-AUC (Higher is Better)")
ax.set_xticks(x)
ax.set_xticklabels(["M0", "M1", "M2", "M3", "M4"])
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.2)

fig.suptitle(
    "Predicting LLM Recommendation Failure Risk",
    fontsize=15,
    fontweight="bold"
)

fig.text(
    0.5, 0.01,
    "N = 73 | 10 × 5-fold stratified cross-validation | "
    "Error bars: ±1 SD across repeats",
    ha="center",
    fontsize=10
)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])

plt.savefig(
    "/content/prediction_evaluation_summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Final evaluation figure saved.")

In [ ]:
# Step 39: Risk-Aware Platform Decision Simulation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Resource constraint
N = len(y)
budget_ratio = 0.20
budget = int(np.ceil(N * budget_ratio))

y_array = np.asarray(y)
total_failures = y_array.sum()

rng = np.random.default_rng(2026)

# 2. Random allocation benchmark
random_captures = []

for _ in range(10000):

    selected = rng.choice(
        N, size=budget, replace=False
    )

    random_captures.append(
        y_array[selected].sum()
    )

random_captures = np.array(random_captures)

random_expected = budget * total_failures / N

random_interval = np.quantile(
    random_captures, [0.025, 0.975]
)

# 3. Define allocation strategies
strategy_scores = {
    "S1_Raw_TVD":
        model_df["raw_tvd"].to_numpy(),

    "S2_Baseline_Risk":
        lr_oof_predictions["M0_Baseline"],

    "S3_Corrected_Shift_Risk":
        lr_oof_predictions["M2_Corrected_Shift"],

    "S4_Behavioral_Risk_LR":
        lr_oof_predictions[
            "M3_Behavioral_Heterogeneity"
        ],

    "S5_Behavioral_Risk_RF":
        rf_oof_predictions[
            "M3_Behavioral_Heterogeneity"
        ]
}

# 4. Evaluate allocation
records = [{
    "Strategy": "S0_Random",
    "Selected Users": budget,
    "Failures Captured": random_expected,
    "Capture Rate": budget / N,
    "Precision": total_failures / N,
    "Lift": 1.0
}]

# Reproducible random tie-breaking
tie_breaker = rng.random(N)

for name, scores in strategy_scores.items():

    scores = np.asarray(scores)

    assert len(scores) == N
    assert np.isfinite(scores).all()

    selected = np.lexsort(
        (tie_breaker, -scores)
    )[:budget]

    captured = y_array[selected].sum()

    capture_rate = captured / total_failures

    records.append({
        "Strategy": name,
        "Selected Users": budget,
        "Failures Captured": captured,
        "Capture Rate": capture_rate,
        "Precision": captured / budget,
        "Lift": capture_rate / (budget / N)
    })

decision_df = pd.DataFrame(records)

print("=== RESOURCE CONSTRAINT ===")
print("Total users:", N)
print("Total failures:", total_failures)
print("Available resources:", budget)
print("Actual budget ratio:", round(budget / N, 4))

print("\n=== RANDOM ALLOCATION BENCHMARK ===")
print("Expected failures captured:",
      round(random_expected, 2))
print("95% random allocation interval:",
      random_interval)

print("\n=== PLATFORM DECISION RESULTS ===")
display(decision_df.round(4))

# 5. Visualization
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    ["S0", "S1", "S2", "S3", "S4", "S5"],
    decision_df["Capture Rate"] * 100,
    color=["gray"] + ["#527DA5"] * 5
)

ax.axhspan(
    random_interval[0] / total_failures * 100,
    random_interval[1] / total_failures * 100,
    color="gray",
    alpha=0.12,
    label="95% Random Allocation Range"
)

ax.axhline(
    budget / N * 100,
    color="black",
    linestyle="--",
    label="Random Expected Capture Rate"
)

ax.bar_label(bars, fmt="%.1f%%", padding=3)

ax.set_ylabel("Failure Capture Rate (%)")
ax.set_title(
    "Risk-Aware Resource Allocation "
    "under a Fixed Budget"
)
ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

plt.savefig(
    "/content/risk_aware_decision.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

decision_df.to_csv(
    "/content/platform_decision_results.csv",
    index=False
)

print("\nDecision simulation completed.")

In [ ]:
# Step 40: Figure 2 — Measurement Construction and Validation

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig = plt.figure(figsize=(11, 7.5))

gs = fig.add_gridspec(
    2, 1,
    height_ratios=[1.3, 1],
    hspace=0.35
)

# ==========================================
# Panel A: Measurement Construction
# ==========================================

ax = fig.add_subplot(gs[0])
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis("off")

ax.set_title(
    "A. From Observed Interactions to Behavioral Measures",
    loc="left", fontsize=13, fontweight="bold"
)

blue = "#527DA5"
green = "#77978A"
orange = "#D4A373"

def draw_box(x, y, w, h, text,
             color, fontsize=10, alpha=0.12):

    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.04",
        facecolor=color,
        edgecolor=color,
        alpha=alpha
    )

    ax.add_patch(patch)

    ax.text(
        x+w/2, y+h/2, text,
        ha="center", va="center",
        fontsize=fontsize
    )

def arrow(x1, y1, x2, y2):

    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle="->",
            color="gray",
            lw=1.2
        )
    )

# Illustrative category sequence
categories = ["A", "A", "B", "B", "C", "C"]

positions = [2, 3, 4, 5.5, 6.5, 7.5]

category_colors = {
    "A": blue,
    "B": orange,
    "C": green
}

ax.text(3.4, 5.65, "Early History",
        ha="center", fontsize=11)

ax.text(6.9, 5.65, "Recent History",
        ha="center", fontsize=11)

for x, c in zip(positions, categories):

    draw_box(
        x, 4.65, 0.8, 0.7,
        c, category_colors[c],
        fontsize=13, alpha=0.35
    )

# Held-out target
draw_box(
    9.5, 4.65, 1.4, 0.7,
    "Held-out\nTarget", "gray"
)

ax.plot(
    [5.05, 5.05], [4.5, 5.5],
    "--", color="gray"
)

ax.plot(
    [9.0, 9.0], [4.5, 5.5],
    "--", color="gray"
)

ax.text(
    5.2, 4.2,
    "Pre-target history only",
    ha="center", fontsize=10
)

# Three behavioral measures
measure_boxes = [
    (0.4, "Preference Shift\nRaw TVD = 0.667"),
    (4.4, "Behavioral Persistence\nRepetition = 0.600"),
    (8.4, "Exploration\nNovelty Rate = 0.667")
]

for x, label in measure_boxes:

    draw_box(
        x, 2.65, 3.2, 0.9,
        label, blue
    )

    arrow(5.2, 4.0, x+1.6, 3.6)

# Correction
output_boxes = [
    (0.4, "Subtract permutation null\nExcess TVD"),
    (4.4, "Subtract expected repetition\nExcess Repetition"),
    (8.4, "Direct behavioral measure\nExploration Rate")
]

for x, label in output_boxes:

    draw_box(
        x, 0.9, 3.2, 0.95,
        label, green
    )

    arrow(x+1.6, 2.6, x+1.6, 1.9)

ax.text(
    6, 0.25,
    "Illustrative sequence; A, B and C denote product categories.",
    ha="center", fontsize=9, color="gray"
)

# ==========================================
# Panel B: Empirical Measurement Validation
# ==========================================

ax2 = fig.add_subplot(gs[1])

variables = [
    "raw_tvd",
    "excess_tvd",
    "repetition_rate",
    "excess_repetition_rate"
]

correlations = [
    final_feature_df["category_entropy"].corr(
        final_feature_df[v],
        method="spearman"
    )
    for v in variables
]

labels = [
    "Raw TVD",
    "Excess TVD",
    "Raw Repetition",
    "Excess Repetition"
]

bars = ax2.barh(
    labels,
    correlations,
    color=[blue, green, blue, green],
    height=0.55
)

ax2.invert_yaxis()

ax2.axvline(
    0, color="black", linewidth=0.8
)

for bar, value in zip(bars, correlations):

    ax2.text(
        value + (0.035 if value >= 0 else -0.035),
        bar.get_y() + bar.get_height()/2,
        f"{value:.3f}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=10
    )

ax2.set_xlim(-1.1, 1.1)

ax2.set_xlabel(
    "Spearman Correlation with Category Entropy"
)

ax2.set_title(
    "B. Measurement Validation",
    loc="left", fontsize=13, fontweight="bold"
)

ax2.grid(axis="x", alpha=0.2)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

fig.suptitle(
    "Construction and Validation of Consumer Behavioral Measures",
    fontsize=15,
    fontweight="bold",
    y=0.98
)

fig.text(
    0.5, 0.02,
    f"Data: AgentRecBench Amazon Long-term | "
    f"Measurement sample: N = {len(final_feature_df):,}",
    ha="center", fontsize=10
)

plt.savefig(
    "/content/figure2_data_measurement.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Figure 2 saved successfully.")

In [ ]:
# Step 41: Figure 3 — Empirical Research Workflow

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 5.5))

ax.set_xlim(0, 16)
ax.set_ylim(0, 7)
ax.axis("off")

blue = "#527DA5"
green = "#77978A"
orange = "#D4A373"

def draw_box(x, y, w, h, title, body, color):

    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.05",
        facecolor=color,
        edgecolor=color,
        alpha=0.15,
        linewidth=1.5
    )

    ax.add_patch(patch)

    ax.text(
        x + w/2, y + h - 0.35,
        title,
        ha="center", va="center",
        fontsize=11,
        fontweight="bold"
    )

    ax.text(
        x + w/2, y + h/2 - 0.15,
        body,
        ha="center", va="center",
        fontsize=9.5,
        linespacing=1.5
    )

def arrow(x1, y1, x2, y2):

    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle="->",
            color="#777777",
            lw=1.5
        )
    )

# ==========================================
# Stage 1: Prediction Dataset
# ==========================================

draw_box(
    0.2, 4.1, 3.3, 1.9,
    "Behavioral Features (X)",
    "Baseline characteristics\n"
    "Shift · Persistence · Exploration\n"
    "Pre-target history only",
    blue
)

draw_box(
    0.2, 1.7, 3.3, 1.9,
    "Failure Labels (Y)",
    "Gemini candidate ranking\n"
    "20 candidate products\n"
    "Failure@5",
    blue
)

arrow(1.85, 4.1, 1.85, 3.85)

# Join X and Y
arrow(3.5, 5.05, 4.15, 3.85)
arrow(3.5, 2.65, 4.15, 3.85)

# ==========================================
# Stage 2: Model Comparison
# ==========================================

draw_box(
    4.2, 1.7, 3.5, 4.3,
    "Incremental Model Design",
    "M0: Baseline\n"
    "M1: M0 + Raw TVD\n"
    "M2: M0 + Excess TVD\n"
    "M3: M2 + Behavioral Features\n"
    "M4: M3 + Interactions\n\n"
    "Logistic Regression: M0–M4\n"
    "Random Forest: M0–M3",
    blue
)

arrow(7.7, 3.85, 8.25, 3.85)

# ==========================================
# Stage 3: Evaluation
# ==========================================

draw_box(
    8.3, 1.7, 3.5, 4.3,
    "Predictive Evaluation",
    "Stratified 5-fold CV\n"
    "10 repeated CV runs\n\n"
    "Primary: Log Loss\n"
    "Secondary: ROC-AUC\n"
    "Brier Score · Calibration\n\n"
    "Naive and feature baselines",
    green
)

arrow(11.8, 3.85, 12.35, 3.85)

# ==========================================
# Stage 4: Decision Simulation
# ==========================================

draw_box(
    12.4, 1.7, 3.4, 4.3,
    "Platform Decisions",
    "Fixed resource budget\n"
    "Approximately 20% of users\n\n"
    "Risk-based allocation\n"
    "vs. Random allocation\n\n"
    "Failure Capture Rate\n"
    "Lift",
    orange
)

# Stage labels
stage_labels = [
    (1.85, "01  DATA"),
    (5.95, "02  PREDICTION"),
    (10.05, "03  EVALUATION"),
    (14.1, "04  DECISION")
]

for x, label in stage_labels:

    ax.text(
        x, 6.4, label,
        ha="center",
        fontsize=10,
        fontweight="bold",
        color="#555555"
    )

# Sample information
ax.text(
    1.85, 1.25,
    f"N = {len(y)} users | {int(y.sum())} failures",
    ha="center",
    fontsize=10,
    color="#555555"
)

# Research boundary
ax.text(
    8, 0.55,
    "Decision simulation evaluates failure identification, "
    "not causal intervention effectiveness.",
    ha="center",
    fontsize=10,
    style="italic",
    color="#666666"
)

fig.suptitle(
    "Empirical Framework for Recommendation Failure Prediction",
    fontsize=16,
    fontweight="bold",
    y=0.98
)

plt.tight_layout()

plt.savefig(
    "/content/figure3_empirical_workflow.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    "/content/figure3_empirical_workflow.svg",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Figure 3 saved successfully.")

In [ ]:
# Step 42: Figure 4 — Predictive Performance and Decision Value

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 1. Figure style
# ==========================================

BLUE = "#527DA5"
GREEN = "#77978A"
ORANGE = "#D4A373"
GRAY = "#929292"
DARK = "#333333"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})

# ==========================================
# 2. Repeated CV results
# ==========================================

# Values from Step 38.1
# Mean and SD across 10 repeated CV runs

model_names = ["M0", "M1", "M2", "M3", "M4"]

lr_logloss = np.array([
    0.6800, 0.6850, 0.6894, 0.7112, 0.7462
])

lr_logloss_sd = np.array([
    0.0153, 0.0188, 0.0188, 0.0171, 0.0248
])

rf_logloss = np.array([
    0.7515, 0.7682, 0.7262, 0.7209
])

rf_logloss_sd = np.array([
    0.0219, 0.0235, 0.0290, 0.0317
])

lr_auc = np.array([
    0.5360, 0.5602, 0.5356, 0.4964, 0.4753
])

lr_auc_sd = np.array([
    0.0435, 0.0447, 0.0461, 0.0488, 0.0529
])

rf_auc = np.array([
    0.3990, 0.3792, 0.4626, 0.5223
])

rf_auc_sd = np.array([
    0.0368, 0.0554, 0.0621, 0.0588
])

naive_logloss = 0.6660
naive_auc = 0.5000

# ==========================================
# 3. Decision simulation results
# ==========================================

# Values from Step 39

strategies = [
    "Raw TVD",
    "LR M0",
    "LR M2",
    "LR M3",
    "RF M3"
]

failures_captured = np.array([6, 6, 6, 5, 7])

n_users = 73
n_failures = 28
budget = 15

random_expected = budget * n_failures / n_users

# Empirical central 95% range
# from 10,000 random allocations
random_lower = 2
random_upper = 9

# ==========================================
# 4. Create figure
# ==========================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 5.3),
    gridspec_kw={"width_ratios": [1, 1, 1.15]}
)

x = np.arange(len(model_names))
width = 0.34

error_style = {
    "ecolor": DARK,
    "elinewidth": 1,
    "capsize": 3,
    "capthick": 1
}

# ==========================================
# Panel A: Log Loss
# ==========================================

ax = axes[0]

ax.bar(
    x - width/2,
    lr_logloss,
    width,
    yerr=lr_logloss_sd,
    color=BLUE,
    label="Logistic Regression",
    error_kw=error_style
)

ax.bar(
    x[:4] + width/2,
    rf_logloss,
    width,
    yerr=rf_logloss_sd,
    color=GREEN,
    label="Random Forest",
    error_kw=error_style
)

ax.axhline(
    naive_logloss,
    color=ORANGE,
    linestyle="--",
    linewidth=2,
    label="Naive baseline"
)

ax.set_title(
    "A. Probabilistic Prediction",
    loc="left",
    pad=15
)

ax.set_ylabel("Mean Log Loss")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylim(0, 0.88)

ax.text(
    0.98, naive_logloss + 0.015,
    "Naive = 0.666",
    transform=ax.get_yaxis_transform(),
    ha="right",
    color="#9B6A35",
    fontsize=9
)

ax.text(
    0.03, 0.96,
    "Lower is better",
    transform=ax.transAxes,
    va="top",
    fontsize=9,
    color=GRAY
)

ax.grid(axis="y", alpha=0.15)
ax.set_axisbelow(True)

# ==========================================
# Panel B: ROC-AUC
# ==========================================

ax = axes[1]

ax.bar(
    x - width/2,
    lr_auc,
    width,
    yerr=lr_auc_sd,
    color=BLUE,
    error_kw=error_style
)

ax.bar(
    x[:4] + width/2,
    rf_auc,
    width,
    yerr=rf_auc_sd,
    color=GREEN,
    error_kw=error_style
)

ax.axhline(
    naive_auc,
    color=ORANGE,
    linestyle="--",
    linewidth=2
)

ax.set_title(
    "B. Risk Discrimination",
    loc="left",
    pad=15
)

ax.set_ylabel("Mean ROC-AUC")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylim(0, 1)

ax.text(
    0.98, naive_auc + 0.025,
    "Chance = 0.500",
    transform=ax.get_yaxis_transform(),
    ha="right",
    color="#9B6A35",
    fontsize=9
)

ax.text(
    0.03, 0.96,
    "Higher is better",
    transform=ax.transAxes,
    va="top",
    fontsize=9,
    color=GRAY
)

ax.grid(axis="y", alpha=0.15)
ax.set_axisbelow(True)

# ==========================================
# Panel C: Decision Value
# ==========================================

ax = axes[2]

positions = np.arange(len(strategies))

# Random allocation reference range
ax.axvspan(
    random_lower,
    random_upper,
    color=GRAY,
    alpha=0.15,
    label="95% random-allocation range"
)

# Expected random allocation
ax.axvline(
    random_expected,
    color=ORANGE,
    linestyle="--",
    linewidth=2
)

bar_colors = [
    GRAY,
    BLUE,
    BLUE,
    BLUE,
    GREEN
]

ax.barh(
    positions,
    failures_captured,
    color=bar_colors,
    height=0.55,
    alpha=0.9
)

for i, value in enumerate(failures_captured):

    ax.text(
        value + 0.15,
        i,
        f"{value}",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

ax.set_yticks(positions)
ax.set_yticklabels(strategies)
ax.invert_yaxis()

ax.set_xlim(0, 10)
ax.set_xticks(np.arange(0, 11, 2))

ax.set_title(
    "C. Platform Decision Value",
    loc="left",
    pad=15
)

ax.set_xlabel("Actual Failures Captured")

ax.text(
    0.98, 0.96,
    "Budget: 15 of 73 users",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    color=DARK
)

ax.text(
    0.98, 0.04,
    f"Random expectation = {random_expected:.2f}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=9,
    color="#9B6A35"
)

ax.grid(axis="x", alpha=0.15)
ax.set_axisbelow(True)

# ==========================================
# 5. Shared title and legend
# ==========================================

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

legend_elements = [

    Patch(
        facecolor=BLUE,
        label="Logistic Regression"
    ),

    Patch(
        facecolor=GREEN,
        label="Random Forest"
    ),

    Line2D(
        [0], [0],
        color=ORANGE,
        linestyle="--",
        linewidth=2,
        label="Naive / Random baseline"
    ),

    Patch(
        facecolor=GRAY,
        alpha=0.15,
        label="95% random-allocation range"
    )
]

fig.suptitle(
    "Predictive Performance and Risk-Aware Decision Value",
    fontsize=17,
    fontweight="bold",
    y=0.98
)

fig.legend(
    handles=legend_elements,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.075),
    ncol=4,
    frameon=False,
    fontsize=9
)

# ==========================================
# 6. Methodological note
# ==========================================

fig.text(
    0.5, 0.025,
    "Panels A–B: mean ± SD across 10 repeated five-fold CV runs. "
    "Panel C: 10,000 random allocations; shaded area indicates "
    "the empirical central 95% range.",
    ha="center",
    fontsize=9,
    color="#666666"
)

plt.tight_layout(
    rect=[0.01, 0.16, 0.99, 0.90],
    w_pad=3
)

# ==========================================
# 7. Export
# ==========================================

plt.savefig(
    "/content/figure4_predictive_decision_results.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    "/content/figure4_predictive_decision_results.svg",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Figure 4 saved successfully.")

In [ ]:
# Step 43: Appendix B — Revised Future Research Roadmap

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# ==========================================
# 1. Figure style
# ==========================================

BLUE = "#527DA5"
GREEN = "#77978A"
ORANGE = "#D4A373"
GRAY = "#929292"
DARK = "#333333"

fig, ax = plt.subplots(figsize=(14, 9))

ax.set_xlim(0, 15)
ax.set_ylim(0, 10)
ax.axis("off")

def box(x, y, w, h, color, alpha=0.12, dashed=False):

    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.04,rounding_size=0.12",
        facecolor=color,
        edgecolor=color,
        linewidth=1.5,
        alpha=alpha,
        linestyle="--" if dashed else "-"
    )

    ax.add_patch(patch)

def text(x, y, content, size=11,
         bold=False, color=DARK):

    ax.text(
        x, y, content,
        ha="center",
        va="center",
        fontsize=size,
        fontweight="bold" if bold else "normal",
        color=color,
        linespacing=1.5
    )

def arrow(x1, y1, x2, y2, dashed=False):

    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle="->",
            color="#888888",
            lw=1.5,
            linestyle="--" if dashed else "-"
        )
    )

# ==========================================
# 2. Research foundation
# ==========================================

box(0.4, 8.05, 14.2, 1.35, GRAY, 0.10)

text(
    7.5, 9.12,
    "RESEARCH FOUNDATION",
    11, True, "#666666"
)

text(4.0, 8.65, "Data and Measurement", 13, True)

text(
    4.0, 8.30,
    "Behavioral representation · Measurement validity",
    10
)

text(11.0, 8.65, "Explanation", 13, True)

text(
    11.0, 8.30,
    "Behavioral associations · Interaction analysis",
    10
)

ax.plot(
    [7.5, 7.5], [8.20, 8.90],
    color=GRAY, alpha=0.35
)

# Foundation supports all three stages
arrow(7.5, 8.05, 7.5, 7.65)

# ==========================================
# 3. Core research stages
# ==========================================

stages = [
    {
        "x": 0.4,
        "color": BLUE,
        "status": "STAGE 01 · CURRENT",
        "keyword": "WHEN & FOR WHOM",
        "title": "Prediction",
        "question":
            "Can consumer behavior predict\n"
            "recommendation failure?",
        "design":
            "Behavioral features\n"
            "Logistic Regression · Random Forest",
        "check":
            "Log Loss · AUC · Calibration"
    },
    {
        "x": 5.3,
        "color": GREEN,
        "status": "STAGE 02 · FUTURE",
        "keyword": "WHY",
        "title": "Causal Inference",
        "question":
            "Do behavioral changes causally\n"
            "affect recommendation performance?",
        "design":
            "Controlled input experiments\n"
            "Causal identification",
        "check":
            "Treatment Effects · Robustness"
    },
    {
        "x": 10.2,
        "color": ORANGE,
        "status": "STAGE 03 · FUTURE",
        "keyword": "HOW",
        "title": "Optimization",
        "question":
            "How can recommendation algorithms\n"
            "adapt to heterogeneous consumers?",
        "design":
            "Adaptive history weighting\n"
            "Personalized recommendation strategies",
        "check":
            "Failure@5 · Quality · Cost"
    }
]

for s in stages:

    x = s["x"]
    c = s["color"]
    w = 4.4
    center = x + w / 2

    box(x, 4.25, w, 3.35, c, 0.14)

    # Colored top accent
    ax.plot(
        [x + 0.12, x + w - 0.12],
        [7.53, 7.53],
        color=c,
        linewidth=5,
        solid_capstyle="round"
    )

    text(center, 7.22, s["status"], 9, True, c)

    text(center, 6.80, s["keyword"], 13, True, c)

    text(center, 6.40, s["title"], 18, True)

    text(center, 5.90, s["question"], 10)

    # Divider
    ax.plot(
        [x + 0.35, x + w - 0.35],
        [5.48, 5.48],
        color=c,
        alpha=0.45
    )

    text(center, 5.23, "RESEARCH DESIGN", 8, True, c)

    text(center, 4.88, s["design"], 9.5)

    text(center, 4.45, s["check"], 9, True)

# Main research progression
arrow(4.8, 5.95, 5.3, 5.95)
arrow(9.7, 5.95, 10.2, 5.95)

# ==========================================
# 4. Current evidence and frontier methods
# ==========================================

# Current evidence
box(0.4, 2.85, 4.4, 1.0, BLUE, 0.08, True)

text(2.6, 3.60, "CURRENT EVIDENCE", 9, True, BLUE)

text(
    2.6, 3.20,
    "No reliable predictive improvement\n"
    "over the naive baseline",
    10
)

arrow(2.6, 4.25, 2.6, 3.85, dashed=True)

# Frontier methods
box(5.3, 2.85, 9.3, 1.0, ORANGE, 0.10)

text(
    9.95, 3.60,
    "FRONTIER METHODS · AGENTIC COMMERCE",
    10, True, "#A47746"
)

text(
    9.95, 3.23,
    "Contextual Bandits  →  Reinforcement Learning",
    12, True
)

text(
    9.95, 2.99,
    "Sequential adaptation · Cumulative reward · Policy robustness",
    9
)

arrow(12.4, 4.25, 12.4, 3.85)

# ==========================================
# 5. Scientific advance cycle
# ==========================================

box(0.4, 0.55, 14.2, 1.95, GRAY, 0.08)

text(
    7.5, 2.25,
    "EVIDENCE-DRIVEN SCIENTIFIC ADVANCE CYCLE",
    11, True
)

cycle_labels = [
    "Design",
    "Train / Estimate",
    "Test",
    "Compare",
    "Revise"
]

cycle_x = [0.8, 3.65, 6.5, 9.35, 12.2]

for x, label in zip(cycle_x, cycle_labels):

    box(x, 1.35, 2.0, 0.55, BLUE, 0.13)

    text(x + 1.0, 1.625, label, 10, True)

for i in range(4):

    arrow(
        cycle_x[i] + 2.0, 1.625,
        cycle_x[i + 1], 1.625
    )

# Revision loop
ax.plot(
    [13.2, 13.2, 1.8],
    [1.35, 1.05, 1.05],
    color=GRAY,
    linestyle="--",
    linewidth=1.3
)

arrow(1.8, 1.05, 1.8, 1.35, dashed=True)

# Current findings trigger revision
arrow(2.6, 2.85, 2.6, 2.50, dashed=True)

text(
    7.5, 0.77,
    "Fresh evidence: larger labeled samples · Richer features · "
    "External validation · Controlled experiments",
    10
)

# ==========================================
# 6. Research boundary
# ==========================================

text(
    7.5, 0.15,
    "Research progression depends on credible evidence; "
    "predictive associations alone do not establish causality.",
    9, False, "#777777"
)

# ==========================================
# 7. Title and export
# ==========================================

fig.suptitle(
    "From Failure Prediction to Adaptive Recommendation",
    fontsize=20,
    fontweight="bold",
    y=0.98
)

ax.text(
    7.5, 9.72,
    "Future Research Roadmap",
    ha="center",
    fontsize=12,
    color="#666666"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])

plt.savefig(
    "/content/appendixB_future_research_roadmap_v2.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    "/content/appendixB_future_research_roadmap_v2.svg",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Revised Appendix B roadmap saved successfully.")